# 04 _ LightGBM Forecasting Model

**Project:** Corporacion Favorita Store Sales Forecasting  
**Project type:** Data Science / MLOps Portfolio Project  
**Notebook:** `notebooks/04_lightgbm_forecasting_model.ipynb`

---

## 1.Purpose

This notebook trains the first main machine learning model for the forecasting system.

The goal is to evaluate whether LightGBM improves the best simple baseline while preserving the operational prediction grain required by store replenishment.


# 2.Notebook Objective

The objective of this notebook is to train and evaluate `lightgbm_v1` using the feature datasets created in notebook 03.

The notebook will:

- Load the modeling datasets and baseline artifacts.
- Validate feature inputs and prediction grain.
- Define a safe feature policy for LightGBM.
- Run a small smoke test before full training.
- Train the validation model and full serving model.
- Evaluate validation performance against the best baseline.
- Analyze feature importance and business segment errors.
- Save model files, configs, predictions, metrics, and submission artifacts.


# 3.Scope.

## Scope.

This notebook is focused on the first production-oriented tabular model candidate.

It includes:

- LightGBM training.
- Baseline comparison.
- Overfitting and stability review.
- Feature importance analysis.
- Segment-level error analysis.
- Kaggle-style submission generation.

It does not include:

- Prophet modeling.
- MLflow tracking.
- API development.
- Docker deployment.
- Full hyperparameter optimization.
- Rebuilding the data cleaning pipeline.


# 4.Imports and Path Setup

This section prepares the notebook environment, detects the project root dynamically, defines LightGBM paths, validates required input artifacts, and sets local CPU-based execution controls.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import platform
import warnings
import gc
import time

import numpy as np
import pandas as pd

from IPython.display import display

try:
    import pyarrow as pa
except ImportError:
    pa = None

try:
    import lightgbm as lgb
except ImportError as error:
    raise ImportError(
        "LightGBM is required for this notebook. "
        "Install it with: pip install lightgbm"
    ) from error

try:
    import tqdm as tqdm_package
    from tqdm.auto import tqdm

    TQDM_AVAILABLE = True
    TQDM_VERSION = tqdm_package.__version__
except ImportError:
    tqdm = None
    TQDM_AVAILABLE = False
    TQDM_VERSION = "not installed"


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 140)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

SEED = 42
np.random.seed(SEED)

MODEL_VERSION = "lightgbm_v1"
USE_GPU = False
NUM_THREADS = 3
TRAINING_PROGRESS_PERIOD = 50
EARLY_STOPPING_ROUNDS = 50


class LightGBMTqdmCallback:
    """Display a tqdm progress bar for LightGBM training iterations."""

    order = 5
    before_iteration = False

    def __init__(
        self,
        total_rounds,
        description,
        metric_update_period=TRAINING_PROGRESS_PERIOD,
        enabled=TQDM_AVAILABLE,
    ):
        self.total_rounds = int(total_rounds) if total_rounds is not None else None
        self.description = description
        self.metric_update_period = max(1, int(metric_update_period))
        self.enabled = bool(enabled and TQDM_AVAILABLE and tqdm is not None)
        self.progress_bar = None
        self.current_iteration = 0

    def __call__(self, env):
        if not self.enabled:
            return

        if self.progress_bar is None:
            total_rounds = self.total_rounds

            if total_rounds is None:
                total_rounds = env.end_iteration - env.begin_iteration

            self.progress_bar = tqdm(
                total=total_rounds,
                desc=self.description,
                unit="round",
                leave=True,
            )

        completed_iteration = env.iteration - env.begin_iteration + 1
        update_by = completed_iteration - self.current_iteration

        if update_by > 0:
            self.progress_bar.update(update_by)
            self.current_iteration = completed_iteration

        if env.evaluation_result_list and (
            completed_iteration % self.metric_update_period == 0
            or completed_iteration >= self.progress_bar.total
        ):
            metrics = {}

            for data_name, metric_name, metric_value, *_ in env.evaluation_result_list:
                if isinstance(metric_value, (int, float, np.floating)):
                    metrics[f"{data_name}_{metric_name}"] = f"{metric_value:.5f}"

            if metrics:
                self.progress_bar.set_postfix(metrics)

        if self.progress_bar.total and completed_iteration >= self.progress_bar.total:
            self.close()

    def close(self):
        if self.progress_bar is not None:
            self.progress_bar.close()
            self.progress_bar = None


def find_project_root(
    start_path=None, required_dirs=("data", "notebooks", "reports", "models")
):
    """
    Find the project root by walking upward from the current working directory.

    The project root is identified as the nearest parent directory containing
    the required project folders.
    """
    current_path = Path.cwd() if start_path is None else Path(start_path)
    current_path = current_path.resolve()

    candidate_paths = [current_path] + list(current_path.parents)

    for candidate_path in candidate_paths:
        if all((candidate_path / directory).exists() for directory in required_dirs):
            return candidate_path

    raise FileNotFoundError(
        "Project root could not be detected. "
        f"Expected a parent directory containing: {required_dirs}. "
        f"Current working directory: {current_path}"
    )


def format_file_size_mb(path):
    """Return file size in MB if the file exists."""
    path = Path(path)
    if not path.exists():
        return np.nan
    return round(path.stat().st_size / (1024**2), 2)


def dataframe_memory_mb(df):
    """Return DataFrame memory usage in MB."""
    return round(df.memory_usage(deep=True).sum() / (1024**2), 2)


def dataframe_overview(df, dataset_name):
    """Create a compact technical overview for a DataFrame."""
    overview = {
        "dataset": dataset_name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": dataframe_memory_mb(df),
    }

    if "date" in df.columns:
        overview["start_date"] = df["date"].min()
        overview["end_date"] = df["date"].max()

    if "store_nbr" in df.columns:
        overview["stores"] = df["store_nbr"].nunique()

    if "family" in df.columns:
        overview["families"] = df["family"].nunique()

    if "sales" in df.columns:
        overview["has_target"] = True
    else:
        overview["has_target"] = False

    return overview


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = DATA_DIR / "features"
PREDICTIONS_DIR = DATA_DIR / "predictions"

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"

MODELS_DIR = PROJECT_ROOT / "models"
BASELINES_MODELS_DIR = MODELS_DIR / "baselines"
LIGHTGBM_MODELS_DIR = MODELS_DIR / "lightgbm"

LIGHTGBM_MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_TRAIN_FEATURES_PATH = FEATURES_DIR / "baseline_train_features.parquet"
BASELINE_VALID_FEATURES_PATH = FEATURES_DIR / "baseline_valid_features.parquet"
BASELINE_TEST_FEATURES_PATH = FEATURES_DIR / "baseline_test_features.parquet"
BASELINE_FULL_FEATURES_PATH = FEATURES_DIR / "baseline_full_features.parquet"

BASELINE_VALIDATION_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / "baseline_validation_predictions.parquet"
)

BASELINE_METRICS_PATH = REPORTS_TABLES_DIR / "baseline_metrics.csv"
BASELINE_LEADERBOARD_PATH = BASELINES_MODELS_DIR / "baseline_leaderboard.csv"
BASELINE_CONFIG_PATH = BASELINES_MODELS_DIR / "baseline_config.json"

LIGHTGBM_MODEL_PATH = LIGHTGBM_MODELS_DIR / f"{MODEL_VERSION}_model.txt"
LIGHTGBM_CONFIG_PATH = LIGHTGBM_MODELS_DIR / f"{MODEL_VERSION}_config.json"
LIGHTGBM_FEATURE_LIST_PATH = LIGHTGBM_MODELS_DIR / f"{MODEL_VERSION}_feature_list.json"

LIGHTGBM_VALIDATION_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"{MODEL_VERSION}_validation_predictions.parquet"
)
LIGHTGBM_TEST_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"{MODEL_VERSION}_test_predictions.parquet"
)
LIGHTGBM_SUBMISSION_PATH = PREDICTIONS_DIR / f"{MODEL_VERSION}_submission.csv"

LIGHTGBM_METRICS_PATH = REPORTS_TABLES_DIR / f"{MODEL_VERSION}_metrics.csv"
LIGHTGBM_BASELINE_COMPARISON_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_baseline_comparison.csv"
)
LIGHTGBM_FEATURE_IMPORTANCE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_feature_importance.csv"
)

LIGHTGBM_METRICS_BY_DATE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_metrics_by_date.csv"
)
LIGHTGBM_METRICS_BY_STORE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_metrics_by_store.csv"
)
LIGHTGBM_METRICS_BY_FAMILY_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_metrics_by_family.csv"
)
LIGHTGBM_METRICS_BY_PROMOTION_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_metrics_by_promotion.csv"
)
LIGHTGBM_METRICS_BY_HORIZON_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_metrics_by_horizon_day.csv"
)

required_input_files = {
    "BASELINE_TRAIN_FEATURES_PATH": BASELINE_TRAIN_FEATURES_PATH,
    "BASELINE_VALID_FEATURES_PATH": BASELINE_VALID_FEATURES_PATH,
    "BASELINE_TEST_FEATURES_PATH": BASELINE_TEST_FEATURES_PATH,
    "BASELINE_FULL_FEATURES_PATH": BASELINE_FULL_FEATURES_PATH,
    "BASELINE_VALIDATION_PREDICTIONS_PATH": BASELINE_VALIDATION_PREDICTIONS_PATH,
    "BASELINE_METRICS_PATH": BASELINE_METRICS_PATH,
    "BASELINE_LEADERBOARD_PATH": BASELINE_LEADERBOARD_PATH,
    "BASELINE_CONFIG_PATH": BASELINE_CONFIG_PATH,
}

input_file_inventory = pd.DataFrame(
    [
        {
            "file_key": file_key,
            "file_name": path.name,
            "expected_location": str(path.parent.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "file_size_mb": format_file_size_mb(path),
            "path": path,
        }
        for file_key, path in required_input_files.items()
    ]
)

display(input_file_inventory)

missing_input_files = input_file_inventory.loc[~input_file_inventory["exists"]]

if not missing_input_files.empty:
    raise FileNotFoundError(
        "Missing required input files from notebook 03:\n"
        + "\n".join(
            [
                f"- {row.file_key}: {row.path}"
                for row in missing_input_files.itertuples(index=False)
            ]
        )
    )

dependency_versions = pd.DataFrame(
    [
        {"package": "python", "version": platform.python_version()},
        {"package": "pandas", "version": pd.__version__},
        {"package": "numpy", "version": np.__version__},
        {
            "package": "pyarrow",
            "version": pa.__version__ if pa is not None else "not installed",
        },
        {"package": "lightgbm", "version": lgb.__version__},
        {"package": "tqdm", "version": TQDM_VERSION},
    ]
)

execution_settings = pd.DataFrame(
    [
        {"setting": "model_version", "value": MODEL_VERSION},
        {"setting": "project_root", "value": PROJECT_ROOT},
        {"setting": "use_gpu", "value": USE_GPU},
        {"setting": "num_threads", "value": NUM_THREADS},
        {"setting": "seed", "value": SEED},
        {"setting": "training_progress_period", "value": TRAINING_PROGRESS_PERIOD},
        {"setting": "early_stopping_rounds", "value": EARLY_STOPPING_ROUNDS},
        {"setting": "tqdm_available", "value": TQDM_AVAILABLE},
        {"setting": "operating_system", "value": platform.platform()},
        {"setting": "cpu_count", "value": os.cpu_count()},
    ]
)

output_directories = pd.DataFrame(
    [
        {
            "directory": "LIGHTGBM_MODELS_DIR",
            "path": LIGHTGBM_MODELS_DIR,
            "exists": LIGHTGBM_MODELS_DIR.exists(),
        },
        {
            "directory": "PREDICTIONS_DIR",
            "path": PREDICTIONS_DIR,
            "exists": PREDICTIONS_DIR.exists(),
        },
        {
            "directory": "REPORTS_TABLES_DIR",
            "path": REPORTS_TABLES_DIR,
            "exists": REPORTS_TABLES_DIR.exists(),
        },
        {
            "directory": "REPORTS_FIGURES_DIR",
            "path": REPORTS_FIGURES_DIR,
            "exists": REPORTS_FIGURES_DIR.exists(),
        },
    ]
)

display(dependency_versions)
display(execution_settings)
display(output_directories)


.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,file_key,file_name,expected_location,exists,file_size_mb,path
0,BASELINE_TRAIN_FEATURES_PATH,baseline_train_features.parquet,data\features,True,64.3100,....
1,BASELINE_VALID_FEATURES_PATH,baseline_valid_features.parquet,data\features,True,0.8000,....
2,BASELINE_TEST_FEATURES_PATH,baseline_test_features.parquet,data\features,True,0.7200,....
3,BASELINE_FULL_FEATURES_PATH,baseline_full_features.parquet,data\features,True,65.0300,....
4,BASELINE_VALIDATION_PREDICTIONS_PATH,baseline_validation_predictions.parquet,data\predictions,True,0.3300,....
5,BASELINE_METRICS_PATH,baseline_metrics.csv,reports\tables,True,0.0000,....
6,BASELINE_LEADERBOARD_PATH,baseline_leaderboard.csv,models\baselines,True,0.0000,....
7,BASELINE_CONFIG_PATH,baseline_config.json,models\baselines,True,0.0000,....


,package,version
0,python,3.11.9
1,pandas,3.0.3
2,numpy,2.4.5
3,pyarrow,24.0.0
4,lightgbm,4.6.0
5,tqdm,4.67.3


,setting,value
0,model_version,lightgbm_v1
1,project_root,.
2,use_gpu,False
3,num_threads,3
4,seed,42
5,training_progress_period,50
6,early_stopping_rounds,50
7,tqdm_available,True
8,operating_system,Windows-10-10.0.19045-SP0
9,cpu_count,8


,directory,path,exists
0,LIGHTGBM_MODELS_DIR,....,True
1,PREDICTIONS_DIR,....,True
2,REPORTS_TABLES_DIR,....,True
3,REPORTS_FIGURES_DIR,....,True


### Section conclusion

The notebook environment is configured with dynamic project paths, reproducible execution settings, CPU-based LightGBM training controls, and validated input artifacts from notebook 03.

The required feature datasets, baseline metrics, baseline predictions, and baseline configuration are available. LightGBM output directories are ready for model artifacts, predictions, metrics, and future experiment tracking.


# 5.Load Modeling Datasets and Baseline Artifacts
This section loads the modeling dataset and baseline artifacts created in notebook 03.

To keep memory usage controlled, only the train, validation, and test feature datasets are loaded at this
stage. The full training dataset will only be loaded later if final Kaggle submission is generated.

The loaded artifacts allow this notebook to train LightGBM v1 and compare it fairly against the baseline models. 


In [2]:
import gc
import time


def read_parquet_with_log(path, dataset_name):
    """Read a parquet file and print a compact loading log."""
    start_time = time.perf_counter()
    print(f"Loading {dataset_name}...")

    df = pd.read_parquet(path)

    elapsed_time = time.perf_counter() - start_time
    memory_mb = dataframe_memory_mb(df)

    print(
        f"Loaded {dataset_name}: "
        f"{df.shape[0]:,} rows, {df.shape[1]:,} columns, "
        f"{memory_mb:,.2f} MB, {elapsed_time:,.2f} seconds"
    )

    return df


train_features = read_parquet_with_log(
    BASELINE_TRAIN_FEATURES_PATH,
    "train_features",
)

valid_features = read_parquet_with_log(
    BASELINE_VALID_FEATURES_PATH,
    "valid_features",
)

test_features = read_parquet_with_log(
    BASELINE_TEST_FEATURES_PATH,
    "test_features",
)

baseline_validation_predictions = read_parquet_with_log(
    BASELINE_VALIDATION_PREDICTIONS_PATH,
    "baseline_validation_predictions",
)

baseline_metrics = pd.read_csv(BASELINE_METRICS_PATH)
baseline_leaderboard = pd.read_csv(BASELINE_LEADERBOARD_PATH)

with open(BASELINE_CONFIG_PATH, "r", encoding="utf-8") as file:
    baseline_config = json.load(file)

for df in [
    train_features,
    valid_features,
    test_features,
    baseline_validation_predictions,
]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

dataset_overview = pd.DataFrame(
    [
        dataframe_overview(train_features, "train_features"),
        dataframe_overview(valid_features, "valid_features"),
        dataframe_overview(test_features, "test_features"),
        dataframe_overview(
            baseline_validation_predictions,
            "baseline_validation_predictions",
        ),
    ]
)

display(dataset_overview)

artifact_overview = pd.DataFrame(
    [
        {
            "artifact": "baseline_metrics",
            "rows": len(baseline_metrics),
            "columns": baseline_metrics.shape[1],
            "memory_mb": dataframe_memory_mb(baseline_metrics),
        },
        {
            "artifact": "baseline_leaderboard",
            "rows": len(baseline_leaderboard),
            "columns": baseline_leaderboard.shape[1],
            "memory_mb": dataframe_memory_mb(baseline_leaderboard),
        },
        {
            "artifact": "baseline_config",
            "rows": np.nan,
            "columns": len(baseline_config),
            "memory_mb": np.nan,
        },
    ]
)

display(artifact_overview)

required_train_valid_columns = {"date", "store_nbr", "family", "sales"}
required_test_columns = {"date", "store_nbr", "family"}

missing_train_columns = required_train_valid_columns - set(train_features.columns)
missing_valid_columns = required_train_valid_columns - set(valid_features.columns)
missing_test_columns = required_test_columns - set(test_features.columns)

if missing_train_columns:
    raise ValueError(
        f"Missing columns in train_features: {sorted(missing_train_columns)}"
    )

if missing_valid_columns:
    raise ValueError(
        f"Missing columns in valid_features: {sorted(missing_valid_columns)}"
    )

if missing_test_columns:
    raise ValueError(
        f"Missing columns in test_features: {sorted(missing_test_columns)}"
    )

if "sales" in test_features.columns:
    raise ValueError(
        "The test feature dataset should not contain the target column 'sales'."
    )

date_summary = pd.DataFrame(
    [
        {
            "dataset": "train_features",
            "start_date": train_features["date"].min(),
            "end_date": train_features["date"].max(),
            "unique_dates": train_features["date"].nunique(),
        },
        {
            "dataset": "valid_features",
            "start_date": valid_features["date"].min(),
            "end_date": valid_features["date"].max(),
            "unique_dates": valid_features["date"].nunique(),
        },
        {
            "dataset": "test_features",
            "start_date": test_features["date"].min(),
            "end_date": test_features["date"].max(),
            "unique_dates": test_features["date"].nunique(),
        },
    ]
)

display(date_summary)

print("Baseline metrics columns:")
print(list(baseline_metrics.columns))

print("\nBaseline leaderboard columns:")
print(list(baseline_leaderboard.columns))

display(baseline_leaderboard.head())

gc.collect()


Loading train_features...
Loaded train_features: 2,972,376 rows, 44 columns, 744.15 MB, 5.65 seconds
Loading valid_features...
Loaded valid_features: 28,512 rows, 44 columns, 7.14 MB, 0.42 seconds
Loading test_features...
Loaded test_features: 28,512 rows, 43 columns, 6.92 MB, 0.10 seconds
Loading baseline_validation_predictions...
Loaded baseline_validation_predictions: 28,512 rows, 12 columns, 2.20 MB, 0.12 seconds


,dataset,rows,columns,memory_mb,start_date,end_date,stores,families,has_target
0,train_features,2972376,44,744.1500,2013-01-01,2017-07-30,54,33,True
1,valid_features,28512,44,7.1400,2017-07-31,2017-08-15,54,33,True
2,test_features,28512,43,6.9200,2017-08-16,2017-08-31,54,33,False
3,baseline_validation_predictions,28512,12,2.2000,2017-07-31,2017-08-15,54,33,True


,artifact,rows,columns,memory_mb
0,baseline_metrics,5.0000,12,0.0000
1,baseline_leaderboard,5.0000,12,0.0000
2,baseline_config,NaN,19,NaN


,dataset,start_date,end_date,unique_dates
0,train_features,2013-01-01,2017-07-30,1668
1,valid_features,2017-07-31,2017-08-15,16
2,test_features,2017-08-16,2017-08-31,16


Baseline metrics columns:
['baseline', 'description', 'rmsle', 'mae', 'wape', 'bias', 'total_bias_pct', 'actual_total_sales', 'predicted_total_sales', 'rmsle_rank', 'mae_rank', 'wape_rank']

Baseline leaderboard columns:
['baseline', 'description', 'rmsle', 'mae', 'wape', 'bias', 'total_bias_pct', 'actual_total_sales', 'predicted_total_sales', 'rmsle_rank', 'mae_rank', 'wape_rank']


,baseline,description,rmsle,mae,wape,bias,total_bias_pct,actual_total_sales,predicted_total_sales,rmsle_rank,mae_rank,wape_rank
0,baseline_rolling_mean_28_origin,Uses the average sales of the last 28 days bef...,0.5216,97.2231,0.2081,6.2760,0.0134,"13,319,179.7816","13,498,121.0000",1,2,2
1,baseline_last_7_day_pattern,Repeats the last known weekly pattern before t...,0.6170,96.5354,0.2067,1.9146,0.0041,"13,319,179.7816","13,373,770.0000",2,1,1
2,baseline_last_observed,Repeats the last observed sales value before t...,0.6595,207.5058,0.4442,163.4701,0.3499,"13,319,179.7816","17,980,040.0000",3,4,4
3,baseline_store_family_weekday_avg,"Uses the historical average by store, family a...",0.6916,142.7926,0.3057,-113.5773,-0.2431,"13,319,179.7816","10,080,863.0000",4,3,3
4,baseline_zero,Predicts zero sales for every row.,4.4195,467.1429,1.0000,-467.1429,-1.0000,"13,319,179.7816",0.0000,5,5,5


40

# 6.Feature Policy for LightGBM v1

This section defines the feature policy for the first LightGBM model.

The model should only use variables that are available at prediction time or historical features built with a safe temporal shift.

The target column is `sales`, and it is excluded from the feature matrix. Identifier columns, raw dates, and any future-only variables are also excluded.

Categorical variables are kept as categorical features instead of being one-hot encoded. This keeps the model compact and reduces memory usage.


In [3]:
TARGET_COLUMN = "sales"

BASE_EXCLUDED_COLUMNS = [
    "id",
    "date",
    TARGET_COLUMN,
]

POTENTIAL_LEAKAGE_COLUMNS = [
    "transactions",
]

RECOMMENDED_FEATURES = [
    # Store and product structure
    "store_nbr",
    "family",
    "city",
    "state",
    "store_type",
    "cluster",
    # Promotion
    "onpromotion",
    # Calendar
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "horizon_day",
    # Safe historical sales features
    "sales_lag_16",
    "sales_lag_21",
    "sales_lag_28",
    "sales_rolling_mean_7_shift_16",
    "sales_rolling_mean_14_shift_16",
    "sales_rolling_mean_28_shift_16",
    # Oil
    "dcoilwtico",
    "oil_row_exists_in_raw",
    "dcoilwtico_missing_before_imputation",
    "dcoilwtico_was_imputed",
    # Calendar events
    "calendar_event_count",
    "transferred_event_count",
    "unique_locale_names",
    "non_transferred_event_count",
    "is_calendar_event",
    "has_non_transferred_event",
    "has_transferred_event",
]


PREFIX_FEATURES = [
    "holiday_type_",
    "holiday_locale_",
]


available_columns = set(train_features.columns)

prefix_features = sorted(
    [
        column
        for column in train_features.columns
        if any(column.startswith(prefix) for prefix in PREFIX_FEATURES)
    ]
)

recommended_existing_features = [
    column for column in RECOMMENDED_FEATURES if column in available_columns
]

recommended_missing_features = [
    column for column in RECOMMENDED_FEATURES if column not in available_columns
]

leakage_existing_columns = [
    column for column in POTENTIAL_LEAKAGE_COLUMNS if column in available_columns
]

excluded_existing_columns = [
    column
    for column in BASE_EXCLUDED_COLUMNS + POTENTIAL_LEAKAGE_COLUMNS
    if column in available_columns
]

model_features = sorted(
    list(dict.fromkeys(recommended_existing_features + prefix_features))
)

unexpected_model_features = [
    column for column in model_features if column in excluded_existing_columns
]

if TARGET_COLUMN not in train_features.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' is missing from train_features.")

if TARGET_COLUMN not in valid_features.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' is missing from valid_features.")

if TARGET_COLUMN in test_features.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' should not exist in test_features."
    )

if unexpected_model_features:
    raise ValueError(
        "Some excluded columns were selected as model features: "
        f"{unexpected_model_features}"
    )

missing_in_valid = sorted(set(model_features) - set(valid_features.columns))
missing_in_test = sorted(set(model_features) - set(test_features.columns))

if missing_in_valid:
    raise ValueError(f"Model features missing in valid_features: {missing_in_valid}")

if missing_in_test:
    raise ValueError(f"Model features missing in test_features: {missing_in_test}")

feature_policy_summary = pd.DataFrame(
    [
        {
            "category": "recommended_features_existing",
            "count": len(recommended_existing_features),
            "columns": recommended_existing_features,
        },
        {
            "category": "prefix_features_existing",
            "count": len(prefix_features),
            "columns": prefix_features,
        },
        {
            "category": "recommended_features_missing",
            "count": len(recommended_missing_features),
            "columns": recommended_missing_features,
        },
        {
            "category": "excluded_existing_columns",
            "count": len(excluded_existing_columns),
            "columns": excluded_existing_columns,
        },
        {
            "category": "potential_leakage_existing_columns",
            "count": len(leakage_existing_columns),
            "columns": leakage_existing_columns,
        },
        {
            "category": "final_model_features",
            "count": len(model_features),
            "columns": model_features,
        },
    ]
)

display(feature_policy_summary)

model_feature_inventory = pd.DataFrame(
    {
        "feature": model_features,
        "train_dtype": [str(train_features[column].dtype) for column in model_features],
        "valid_dtype": [str(valid_features[column].dtype) for column in model_features],
        "test_dtype": [str(test_features[column].dtype) for column in model_features],
        "missing_train_pct": [
            train_features[column].isna().mean() for column in model_features
        ],
        "missing_valid_pct": [
            valid_features[column].isna().mean() for column in model_features
        ],
        "missing_test_pct": [
            test_features[column].isna().mean() for column in model_features
        ],
    }
)

display(model_feature_inventory)

print(f"Final number of LightGBM v1 features: {len(model_features)}")


,category,count,columns
0,recommended_features_existing,32,"[store_nbr, family, city, state, store_type, c..."
1,prefix_features_existing,9,"[holiday_locale_local, holiday_locale_national..."
2,recommended_features_missing,1,[horizon_day]
3,excluded_existing_columns,3,"[id, date, sales]"
4,potential_leakage_existing_columns,0,[]
5,final_model_features,41,"[calendar_event_count, city, cluster, day_of_m..."


,feature,train_dtype,valid_dtype,test_dtype,missing_train_pct,missing_valid_pct,missing_test_pct
0,calendar_event_count,int64,int64,int64,0.0000,0.0000,0.0000
1,city,string,string,string,0.0000,0.0000,0.0000
2,cluster,int64,int64,int64,0.0000,0.0000,0.0000
3,day_of_month,int16,int16,int16,0.0000,0.0000,0.0000
4,day_of_week,int16,int16,int16,0.0000,0.0000,0.0000
5,dcoilwtico,float64,float64,float64,0.0000,0.0000,0.0000
6,dcoilwtico_missing_before_imputation,bool,bool,bool,0.0000,0.0000,0.0000
7,dcoilwtico_was_imputed,bool,bool,bool,0.0000,0.0000,0.0000
8,family,string,string,string,0.0000,0.0000,0.0000
9,has_non_transferred_event,bool,bool,bool,0.0000,0.0000,0.0000


Final number of LightGBM v1 features: 41


### Section conclusion

The LightGBM v1 feature set has been defined using 41 safe modeling variables.

The target, raw date, identifier columns, and potential future-only leakage variables are excluded. No leakage-prone transaction column is present in the selected feature set.

The `horizon_day` feature is not used as a model input in this version because it is not available as a consistent training feature in the current modeling artifacts. It will be created later only for validation error analysis.

Missing values are limited to early historical lag and rolling features in the training dataset, which is expected for the first available dates in each store-family series. LightGBM can handle these missing values natively.


# 7.Train, Validation and Test Matrix Preparation

This section prepares the modeling matrices for LightGBM v1.

The target variable is transformed using `log1p(sales)` because the competition metric is RMSLE and sales are highly skewed.

Categorical variables are converted to pandas `category` dtype so LightGBM can handle them efficiently without one-hot encoding. This keeps memory usage lower and preserves the structure of store, product, and location features.


In [4]:
categorical_feature_candidates = [
    "store_nbr",
    "family",
    "city",
    "state",
    "store_type",
    "cluster",
    "day_of_week",
    "month",
]

categorical_features = [
    feature for feature in categorical_feature_candidates if feature in model_features
]

numeric_features = [
    feature for feature in model_features if feature not in categorical_features
]

memory_before_dtype_optimization = pd.DataFrame(
    [
        {
            "dataset": "train_features",
            "memory_mb": dataframe_memory_mb(train_features),
        },
        {
            "dataset": "valid_features",
            "memory_mb": dataframe_memory_mb(valid_features),
        },
        {
            "dataset": "test_features",
            "memory_mb": dataframe_memory_mb(test_features),
        },
    ]
)


def align_categorical_features(dataframes, categorical_columns):
    """
    Align categorical feature levels across train, validation, and test datasets.

    Category levels are inferred from all available feature datasets because they
    are structural values known before modeling and do not use the target.
    """
    category_summary = []

    for column in categorical_columns:
        category_values = pd.Index([])

        for dataframe in dataframes:
            category_values = category_values.union(
                pd.Index(dataframe[column].dropna().unique())
            )

        category_values = sorted(category_values.tolist())
        categorical_dtype = pd.CategoricalDtype(
            categories=category_values,
            ordered=False,
        )

        for dataframe in dataframes:
            dataframe[column] = dataframe[column].astype(categorical_dtype)

        category_summary.append(
            {
                "feature": column,
                "n_categories": len(category_values),
                "sample_categories": category_values[:10],
            }
        )

    return pd.DataFrame(category_summary)


categorical_feature_summary = align_categorical_features(
    dataframes=[train_features, valid_features, test_features],
    categorical_columns=categorical_features,
)

memory_after_dtype_optimization = pd.DataFrame(
    [
        {
            "dataset": "train_features",
            "memory_mb": dataframe_memory_mb(train_features),
        },
        {
            "dataset": "valid_features",
            "memory_mb": dataframe_memory_mb(valid_features),
        },
        {
            "dataset": "test_features",
            "memory_mb": dataframe_memory_mb(test_features),
        },
    ]
)

memory_optimization_summary = memory_before_dtype_optimization.merge(
    memory_after_dtype_optimization,
    on="dataset",
    suffixes=("_before", "_after"),
)

memory_optimization_summary["memory_reduction_mb"] = (
    memory_optimization_summary["memory_mb_before"]
    - memory_optimization_summary["memory_mb_after"]
)

memory_optimization_summary["memory_reduction_pct"] = (
    memory_optimization_summary["memory_reduction_mb"]
    / memory_optimization_summary["memory_mb_before"]
)

display(categorical_feature_summary)
display(memory_optimization_summary)

if (train_features[TARGET_COLUMN] < 0).any():
    raise ValueError("Negative values found in train target.")

if (valid_features[TARGET_COLUMN] < 0).any():
    raise ValueError("Negative values found in validation target.")

X_train = train_features.loc[:, model_features]
X_valid = valid_features.loc[:, model_features]
X_test = test_features.loc[:, model_features]

y_train = np.log1p(train_features[TARGET_COLUMN]).astype("float32")
y_valid = np.log1p(valid_features[TARGET_COLUMN]).astype("float32")

if not np.isfinite(y_train).all():
    raise ValueError("Non-finite values found in y_train.")

if not np.isfinite(y_valid).all():
    raise ValueError("Non-finite values found in y_valid.")

matrix_overview = pd.DataFrame(
    [
        {
            "matrix": "X_train",
            "rows": X_train.shape[0],
            "columns": X_train.shape[1],
            "memory_mb": dataframe_memory_mb(X_train),
        },
        {
            "matrix": "X_valid",
            "rows": X_valid.shape[0],
            "columns": X_valid.shape[1],
            "memory_mb": dataframe_memory_mb(X_valid),
        },
        {
            "matrix": "X_test",
            "rows": X_test.shape[0],
            "columns": X_test.shape[1],
            "memory_mb": dataframe_memory_mb(X_test),
        },
        {
            "matrix": "y_train",
            "rows": len(y_train),
            "columns": 1,
            "memory_mb": round(y_train.memory_usage(deep=True) / (1024**2), 2),
        },
        {
            "matrix": "y_valid",
            "rows": len(y_valid),
            "columns": 1,
            "memory_mb": round(y_valid.memory_usage(deep=True) / (1024**2), 2),
        },
    ]
)

target_summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "target": "sales",
            "rows": len(train_features),
            "mean_sales": train_features[TARGET_COLUMN].mean(),
            "median_sales": train_features[TARGET_COLUMN].median(),
            "zero_sales_pct": (train_features[TARGET_COLUMN] == 0).mean(),
            "max_sales": train_features[TARGET_COLUMN].max(),
            "mean_log1p_sales": y_train.mean(),
        },
        {
            "dataset": "valid",
            "target": "sales",
            "rows": len(valid_features),
            "mean_sales": valid_features[TARGET_COLUMN].mean(),
            "median_sales": valid_features[TARGET_COLUMN].median(),
            "zero_sales_pct": (valid_features[TARGET_COLUMN] == 0).mean(),
            "max_sales": valid_features[TARGET_COLUMN].max(),
            "mean_log1p_sales": y_valid.mean(),
        },
    ]
)

display(matrix_overview)
display(target_summary)

print(f"Categorical features used by LightGBM: {categorical_features}")
print(f"Numeric features used by LightGBM: {len(numeric_features)}")


,feature,n_categories,sample_categories
0,store_nbr,54,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
1,family,33,"[AUTOMOTIVE, BABY CARE, BEAUTY, BEVERAGES, BOO..."
2,city,22,"[Ambato, Babahoyo, Cayambe, Cuenca, Daule, El ..."
3,state,16,"[Azuay, Bolivar, Chimborazo, Cotopaxi, El Oro,..."
4,store_type,5,"[A, B, C, D, E]"
5,cluster,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
6,day_of_week,7,"[0, 1, 2, 3, 4, 5, 6]"
7,month,12,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"


,dataset,memory_mb_before,memory_mb_after,memory_reduction_mb,memory_reduction_pct
0,train_features,744.1500,541.4300,202.7200,0.2724
1,valid_features,7.1400,5.2000,1.9400,0.2717
2,test_features,6.9200,4.9800,1.9400,0.2803


,matrix,rows,columns,memory_mb
0,X_train,2972376,41,473.4000
1,X_valid,28512,41,4.5500
2,X_test,28512,41,4.5500
3,y_train,2972376,1,11.3400
4,y_valid,28512,1,0.1100


,dataset,target,rows,mean_sales,median_sales,zero_sales_pct,max_sales,mean_log1p_sales
0,train,sales,2972376,356.7267,11.0000,0.3146,"124,717.0000",2.9196
1,valid,sales,28512,467.1429,30.0000,0.1458,"15,190.0000",3.6274


Categorical features used by LightGBM: ['store_nbr', 'family', 'city', 'state', 'store_type', 'cluster', 'day_of_week', 'month']
Numeric features used by LightGBM: 33


### Section conclusion

The LightGBM modeling matrices are ready.

The target has been transformed with `log1p(sales)`, categorical variables have been aligned across train, validation, and test datasets, and the selected model features are available in all splits.

The model is now ready to be trained using a controlled CPU-based LightGBM configuration.


# 8.LightGBM Smoke Test

Before training the full LightGBM model, this section runs a small smoke test using a sample of the training data.

The goal is not to evaluate final model performance, but to verify that the feature matrix, categorical variables, target transformation, LightGBM parameters, callbacks, and metric functions work correctly.

This is especially useful in a local CPU environment with limited memory.


In [5]:
def clip_forecast_predictions(predictions):
    """Clip forecast predictions to avoid negative sales."""
    return np.clip(np.asarray(predictions), 0, None)


def rmsle_score(y_true, y_pred):
    """Compute Root Mean Squared Logarithmic Error."""
    y_true = np.asarray(y_true)
    y_pred = clip_forecast_predictions(y_pred)

    return np.sqrt(np.mean(np.square(np.log1p(y_pred) - np.log1p(y_true))))


def mae_score(y_true, y_pred):
    """Compute Mean Absolute Error."""
    y_true = np.asarray(y_true)
    y_pred = clip_forecast_predictions(y_pred)

    return np.mean(np.abs(y_true - y_pred))


def wape_score(y_true, y_pred):
    """Compute Weighted Absolute Percentage Error."""
    y_true = np.asarray(y_true)
    y_pred = clip_forecast_predictions(y_pred)

    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        return np.nan

    return np.sum(np.abs(y_true - y_pred)) / denominator


def bias_score(y_true, y_pred):
    """Compute average forecast bias."""
    y_true = np.asarray(y_true)
    y_pred = clip_forecast_predictions(y_pred)

    return np.mean(y_pred - y_true)


def total_bias_pct_score(y_true, y_pred):
    """Compute total forecast bias as percentage of total actual sales."""
    y_true = np.asarray(y_true)
    y_pred = clip_forecast_predictions(y_pred)

    denominator = np.sum(y_true)

    if denominator == 0:
        return np.nan

    return (np.sum(y_pred) - np.sum(y_true)) / denominator


def evaluate_forecast(y_true, y_pred, model_name):
    """Return the main forecasting metrics as a one-row DataFrame."""
    return pd.DataFrame(
        [
            {
                "model": model_name,
                "rows": len(y_true),
                "rmsle": rmsle_score(y_true, y_pred),
                "mae": mae_score(y_true, y_pred),
                "wape": wape_score(y_true, y_pred),
                "bias": bias_score(y_true, y_pred),
                "total_bias_pct": total_bias_pct_score(y_true, y_pred),
                "actual_total_sales": np.sum(y_true),
                "predicted_total_sales": np.sum(clip_forecast_predictions(y_pred)),
            }
        ]
    )


lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 1.0,
    "max_bin": 255,
    "force_col_wise": True,
    "deterministic": True,
    "verbosity": -1,
    "num_threads": NUM_THREADS,
    "seed": SEED,
    "feature_fraction_seed": SEED,
    "bagging_seed": SEED,
    "data_random_seed": SEED,
}

training_config_summary = pd.DataFrame(
    [
        {"setting": "model_version", "value": MODEL_VERSION},
        {"setting": "target_transformation", "value": "log1p(sales)"},
        {"setting": "device", "value": "cpu"},
        {"setting": "num_threads", "value": NUM_THREADS},
        {
            "setting": "training_progress",
            "value": f"tqdm progress bar and log every {TRAINING_PROGRESS_PERIOD} rounds",
        },
        {"setting": "early_stopping_rounds", "value": EARLY_STOPPING_ROUNDS},
        {"setting": "categorical_features", "value": categorical_features},
        {"setting": "n_model_features", "value": len(model_features)},
    ]
)

display(training_config_summary)

if "train_features" in globals():
    released_memory_mb = dataframe_memory_mb(train_features)
    del train_features
    gc.collect()

    print(
        f"Released train_features from memory: approximately {released_memory_mb:,.2f} MB"
    )

SMOKE_TEST_ROWS = min(300_000, len(X_train))
SMOKE_TEST_BOOST_ROUNDS = 300
SMOKE_TEST_EARLY_STOPPING_ROUNDS = 25
SMOKE_TEST_LOG_PERIOD = 25

print(f"Smoke test training rows: {SMOKE_TEST_ROWS:,}")

smoke_train_index = X_train.sample(n=SMOKE_TEST_ROWS, random_state=SEED).index

X_train_smoke = X_train.loc[smoke_train_index]
y_train_smoke = y_train.loc[smoke_train_index]

train_smoke_dataset = lgb.Dataset(
    X_train_smoke,
    label=y_train_smoke,
    categorical_feature=categorical_features,
)

valid_smoke_dataset = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=categorical_features,
    reference=train_smoke_dataset,
)

smoke_start_time = time.perf_counter()
smoke_progress = LightGBMTqdmCallback(
    total_rounds=SMOKE_TEST_BOOST_ROUNDS,
    description="LightGBM smoke test",
    metric_update_period=SMOKE_TEST_LOG_PERIOD,
)

try:
    smoke_model = lgb.train(
        params=lgb_params,
        train_set=train_smoke_dataset,
        num_boost_round=SMOKE_TEST_BOOST_ROUNDS,
        valid_sets=[train_smoke_dataset, valid_smoke_dataset],
        valid_names=["train_smoke", "valid"],
        callbacks=[
            smoke_progress,
            lgb.log_evaluation(period=SMOKE_TEST_LOG_PERIOD),
            lgb.early_stopping(
                stopping_rounds=SMOKE_TEST_EARLY_STOPPING_ROUNDS,
                verbose=True,
            ),
        ],
    )
finally:
    smoke_progress.close()

smoke_training_time_seconds = time.perf_counter() - smoke_start_time

smoke_valid_log_predictions = smoke_model.predict(
    X_valid,
    num_iteration=smoke_model.best_iteration,
)

smoke_valid_predictions = np.expm1(smoke_valid_log_predictions)
smoke_valid_predictions = clip_forecast_predictions(smoke_valid_predictions)

smoke_metrics = evaluate_forecast(
    y_true=valid_features[TARGET_COLUMN].to_numpy(),
    y_pred=smoke_valid_predictions,
    model_name="lightgbm_v1_smoke_test",
)

smoke_training_summary = pd.DataFrame(
    [
        {
            "model": "lightgbm_v1_smoke_test",
            "training_rows": SMOKE_TEST_ROWS,
            "validation_rows": len(X_valid),
            "features": len(model_features),
            "best_iteration": smoke_model.best_iteration,
            "training_time_seconds": round(smoke_training_time_seconds, 2),
            "training_time_minutes": round(smoke_training_time_seconds / 60, 2),
        }
    ]
)

display(smoke_training_summary)
display(smoke_metrics)

del (
    X_train_smoke,
    y_train_smoke,
    train_smoke_dataset,
    valid_smoke_dataset,
    smoke_valid_log_predictions,
    smoke_valid_predictions,
    smoke_model,
    smoke_progress,
)

gc.collect()


,setting,value
0,model_version,lightgbm_v1
1,target_transformation,log1p(sales)
2,device,cpu
3,num_threads,3
4,training_progress,tqdm progress bar and log every 50 rounds
5,early_stopping_rounds,50
6,categorical_features,"[store_nbr, family, city, state, store_type, c..."
7,n_model_features,41


Released train_features from memory: approximately 541.43 MB
Smoke test training rows: 300,000


LightGBM smoke test:   2%|▏         | 7/300 [00:00<00:04, 61.89round/s]

Training until validation scores don't improve for 25 rounds


LightGBM smoke test:  11%|█         | 33/300 [00:00<00:04, 57.37round/s, train_smoke_rmse=1.00901, valid_rmse=0.90043]

...

LightGBM smoke test: 100%|██████████| 300/300 [00:04<00:00, 60.84round/s, train_smoke_rmse=0.46037, valid_rmse=0.43751]


[300]	train_smoke's rmse: 0.460367	valid's rmse: 0.437505
Did not meet early stopping. Best iteration is:
[300]	train_smoke's rmse: 0.460367	valid's rmse: 0.437505


,model,training_rows,validation_rows,features,best_iteration,training_time_seconds,training_time_minutes
0,lightgbm_v1_smoke_test,300000,28512,41,300,6.1400,0.1000


,model,rows,rmsle,mae,wape,bias,total_bias_pct,actual_total_sales,predicted_total_sales
0,lightgbm_v1_smoke_test,28512,0.4375,74.5170,0.1595,-1.8393,-0.0039,"13,319,179.7816","13,266,738.1555"


326

### Section conclusion

The LightGBM smoke test verifies that the model configuration, categorical features, target transformation, callbacks, and evaluation metrics work correctly.

The smoke test is not used as the final model result. Its purpose is to reduce execution risk before training LightGBM v1 on the full training dataset.


# 9.LightGBM v1 Full Training

This section trains the full LightGBM v1 model using the complete training matrix.

The model is trained on the `log1p(sales)` target and evaluated on the temporal validation period. Native LightGBM callbacks are used to show training progress and apply early stopping.

The objective is to create the first full machine learning model candidate for the project.


In [6]:
FULL_TRAINING_BOOST_ROUNDS = 1_200

train_dataset = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=categorical_features,
    free_raw_data=False,
)

valid_dataset = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=categorical_features,
    reference=train_dataset,
    free_raw_data=False,
)

evals_result = {}

full_training_start_time = time.perf_counter()
validation_training_progress = LightGBMTqdmCallback(
    total_rounds=FULL_TRAINING_BOOST_ROUNDS,
    description="LightGBM validation training",
)

try:
    lightgbm_v1_model = lgb.train(
        params=lgb_params,
        train_set=train_dataset,
        num_boost_round=FULL_TRAINING_BOOST_ROUNDS,
        valid_sets=[train_dataset, valid_dataset],
        valid_names=["train", "valid"],
        callbacks=[
            validation_training_progress,
            lgb.log_evaluation(period=TRAINING_PROGRESS_PERIOD),
            lgb.early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose=True,
            ),
            lgb.record_evaluation(evals_result),
        ],
    )
finally:
    validation_training_progress.close()

full_training_time_seconds = time.perf_counter() - full_training_start_time

lightgbm_v1_training_summary = pd.DataFrame(
    [
        {
            "model": MODEL_VERSION,
            "training_rows": X_train.shape[0],
            "validation_rows": X_valid.shape[0],
            "features": X_train.shape[1],
            "best_iteration": lightgbm_v1_model.best_iteration,
            "best_train_rmse_log": evals_result["train"]["rmse"][
                lightgbm_v1_model.best_iteration - 1
            ],
            "best_valid_rmse_log": evals_result["valid"]["rmse"][
                lightgbm_v1_model.best_iteration - 1
            ],
            "training_time_seconds": round(full_training_time_seconds, 2),
            "training_time_minutes": round(full_training_time_seconds / 60, 2),
        }
    ]
)

display(lightgbm_v1_training_summary)

del train_dataset, valid_dataset
gc.collect()


LightGBM validation training:   0%|          | 2/1200 [00:00<01:47, 11.16round/s]

Training until validation scores don't improve for 50 rounds


LightGBM validation training:   4%|▍         | 51/1200 [00:08<03:13,  5.94round/s, train_rmse=0.64237, valid_rmse=0.54003]

[50]	train's rmse: 0.642373	valid's rmse: 0.540032


LightGBM validation training:   8%|▊         | 101/1200 [00:16<02:49,  6.50round/s, train_rmse=0.51366, valid_rmse=0.46651]

...

[1150]	train's rmse: 0.397094	valid's rmse: 0.417068


LightGBM validation training: 100%|██████████| 1200/1200 [02:28<00:00,  8.07round/s, train_rmse=0.39543, valid_rmse=0.41677]


[1200]	train's rmse: 0.395435	valid's rmse: 0.416767
Did not meet early stopping. Best iteration is:
[1174]	train's rmse: 0.396284	valid's rmse: 0.416738


,model,training_rows,validation_rows,features,best_iteration,best_train_rmse_log,best_valid_rmse_log,training_time_seconds,training_time_minutes
0,lightgbm_v1,2972376,28512,41,1174,0.3963,0.4167,161.0400,2.6800


0

### Section conclusion

The full LightGBM v1 model has been trained using the complete training dataset and the temporal validation split.

Training progress and early stopping were controlled through native LightGBM callbacks. The resulting model is now ready for validation prediction, metric evaluation, and comparison against the previous baseline models.


# 10.Validation Predictions and Metric Evaluation

This section generates validation predictions from the full LightGBM v1 model and evaluates them on the original sales scale.

The model was trained on `log1p(sales)`, so predictions are transformed back using `expm1`. Negative predictions are clipped to zero because sales cannot be negative.

The evaluation uses the same forecasting metrics as the baseline notebook: RMSLE, MAE, WAPE, bias, and total bias percentage.


In [7]:
best_iteration = lightgbm_v1_model.best_iteration

if best_iteration is None or best_iteration <= 0:
    best_iteration = FULL_TRAINING_BOOST_ROUNDS

valid_log_predictions = lightgbm_v1_model.predict(
    X_valid,
    num_iteration=best_iteration,
)

valid_predictions = np.expm1(valid_log_predictions)
valid_predictions = clip_forecast_predictions(valid_predictions)

valid_actuals = valid_features[TARGET_COLUMN].to_numpy()

lightgbm_v1_metrics = evaluate_forecast(
    y_true=valid_actuals,
    y_pred=valid_predictions,
    model_name=MODEL_VERSION,
)

training_diagnostics = pd.DataFrame(
    [
        {
            "model": MODEL_VERSION,
            "best_iteration": best_iteration,
            "train_rmse_log": evals_result["train"]["rmse"][best_iteration - 1],
            "valid_rmse_log": evals_result["valid"]["rmse"][best_iteration - 1],
            "rmse_log_gap": (
                evals_result["valid"]["rmse"][best_iteration - 1]
                - evals_result["train"]["rmse"][best_iteration - 1]
            ),
            "rmse_log_gap_pct_vs_train": (
                (
                    evals_result["valid"]["rmse"][best_iteration - 1]
                    - evals_result["train"]["rmse"][best_iteration - 1]
                )
                / evals_result["train"]["rmse"][best_iteration - 1]
            ),
            "num_boost_round_limit": FULL_TRAINING_BOOST_ROUNDS,
            "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
            "training_rounds_completed": len(evals_result["valid"]["rmse"]),
            "early_stopping_triggered": len(evals_result["valid"]["rmse"])
            < FULL_TRAINING_BOOST_ROUNDS,
        }
    ]
)

lightgbm_v1_validation_predictions = valid_features[
    [
        column
        for column in [
            "id",
            "date",
            "store_nbr",
            "family",
            "sales",
            "onpromotion",
        ]
        if column in valid_features.columns
    ]
].copy()

lightgbm_v1_validation_predictions["model"] = MODEL_VERSION
lightgbm_v1_validation_predictions["prediction"] = valid_predictions
lightgbm_v1_validation_predictions["residual"] = (
    lightgbm_v1_validation_predictions["prediction"]
    - lightgbm_v1_validation_predictions["sales"]
)
lightgbm_v1_validation_predictions["absolute_error"] = (
    lightgbm_v1_validation_predictions["residual"].abs()
)
lightgbm_v1_validation_predictions["squared_log_error"] = (
    np.log1p(lightgbm_v1_validation_predictions["prediction"])
    - np.log1p(lightgbm_v1_validation_predictions["sales"])
) ** 2

lightgbm_v1_validation_predictions["absolute_percentage_error_safe"] = np.where(
    lightgbm_v1_validation_predictions["sales"] > 0,
    lightgbm_v1_validation_predictions["absolute_error"]
    / lightgbm_v1_validation_predictions["sales"],
    np.nan,
)

validation_prediction_overview = pd.DataFrame(
    [
        {
            "dataset": "lightgbm_v1_validation_predictions",
            "rows": len(lightgbm_v1_validation_predictions),
            "columns": lightgbm_v1_validation_predictions.shape[1],
            "memory_mb": dataframe_memory_mb(lightgbm_v1_validation_predictions),
            "start_date": lightgbm_v1_validation_predictions["date"].min(),
            "end_date": lightgbm_v1_validation_predictions["date"].max(),
            "actual_total_sales": lightgbm_v1_validation_predictions["sales"].sum(),
            "predicted_total_sales": lightgbm_v1_validation_predictions[
                "prediction"
            ].sum(),
        }
    ]
)

display(training_diagnostics)
display(lightgbm_v1_metrics)
display(validation_prediction_overview)
display(lightgbm_v1_validation_predictions.head())

del valid_log_predictions
gc.collect()


,model,best_iteration,train_rmse_log,valid_rmse_log,rmse_log_gap,rmse_log_gap_pct_vs_train,num_boost_round_limit,early_stopping_rounds,training_rounds_completed,early_stopping_triggered
0,lightgbm_v1,1174,0.3963,0.4167,0.0205,0.0516,1200,50,1200,False


,model,rows,rmsle,mae,wape,bias,total_bias_pct,actual_total_sales,predicted_total_sales
0,lightgbm_v1,28512,0.4167,68.1379,0.1459,-0.3896,-0.0008,"13,319,179.7816","13,308,071.3331"


,dataset,rows,columns,memory_mb,start_date,end_date,actual_total_sales,predicted_total_sales
0,lightgbm_v1_validation_predictions,28512,12,2.5400,2017-07-31,2017-08-15,"13,319,179.7816","13,308,071.3331"


,id,date,store_nbr,family,sales,onpromotion,model,prediction,residual,absolute_error,squared_log_error,absolute_percentage_error_safe
0,2972376,2017-07-31,1,AUTOMOTIVE,8.0000,0,lightgbm_v1,3.5773,-4.4227,4.4227,0.4571,0.5528
1,2972377,2017-07-31,1,BABY CARE,0.0000,0,lightgbm_v1,0.0270,0.0270,0.0270,0.0007,NaN
2,2972378,2017-07-31,1,BEAUTY,3.0000,0,lightgbm_v1,3.1246,0.1246,0.1246,0.0009,0.0415
3,2972379,2017-07-31,1,BEVERAGES,"2,414.0000",24,lightgbm_v1,"2,342.3453",-71.6547,71.6547,0.0009,0.0297
4,2972380,2017-07-31,1,BOOKS,1.0000,0,lightgbm_v1,0.1593,-0.8407,0.8407,0.2974,0.8407


0

### Section conclusion

LightGBM v1 validation predictions have been generated and transformed back to the original sales scale.

The model is now evaluated using the same metrics as the baseline notebook, which allows a fair comparison against the previous forecasting benchmarks.

The validation prediction table also includes residuals and absolute errors, making it ready for baseline comparison and segment-level error analysis.


# 11.Baseline Comparison

This section compares LightGBM v1 against the baseline models created in notebook 03.

The comparison uses the same validation period and the same forecasting metrics: RMSLE, MAE, WAPE, bias, and total bias percentage.

The main benchmark is the best baseline by RMSLE. LightGBM v1 is considered a successful first machine learning model if it improves the best baseline while keeping WAPE and bias under control.


In [8]:
baseline_comparison_metrics = baseline_metrics.copy()

if "model" not in baseline_comparison_metrics.columns:
    possible_model_columns = [
        "baseline",
        "baseline_name",
        "model_name",
        "method",
    ]

    available_model_columns = [
        column
        for column in possible_model_columns
        if column in baseline_comparison_metrics.columns
    ]

    if not available_model_columns:
        raise ValueError(
            "No model identifier column found in baseline_metrics. "
            f"Available columns: {list(baseline_comparison_metrics.columns)}"
        )

    baseline_comparison_metrics = baseline_comparison_metrics.rename(
        columns={available_model_columns[0]: "model"}
    )

if "rows" not in baseline_comparison_metrics.columns:
    baseline_comparison_metrics["rows"] = len(valid_features)

required_metric_columns = [
    "model",
    "rows",
    "rmsle",
    "mae",
    "wape",
    "bias",
    "total_bias_pct",
    "actual_total_sales",
    "predicted_total_sales",
]

missing_metric_columns = [
    column
    for column in required_metric_columns
    if column not in baseline_comparison_metrics.columns
]

if missing_metric_columns:
    raise ValueError(
        "Missing required metric columns for baseline comparison: "
        f"{missing_metric_columns}. "
        f"Available columns: {list(baseline_comparison_metrics.columns)}"
    )

baseline_comparison_metrics = baseline_comparison_metrics[
    required_metric_columns
].copy()

baseline_comparison_metrics["model_type"] = "baseline"

lightgbm_metrics_for_comparison = lightgbm_v1_metrics[required_metric_columns].copy()

lightgbm_metrics_for_comparison["model_type"] = "machine_learning"

model_comparison = pd.concat(
    [
        baseline_comparison_metrics,
        lightgbm_metrics_for_comparison,
    ],
    ignore_index=True,
)

model_comparison["abs_bias"] = model_comparison["bias"].abs()
model_comparison["abs_total_bias_pct"] = model_comparison["total_bias_pct"].abs()

model_comparison = model_comparison.sort_values(
    by=["rmsle", "wape", "abs_total_bias_pct"],
    ascending=[True, True, True],
).reset_index(drop=True)

model_comparison["rmsle_rank"] = (
    model_comparison["rmsle"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

best_baseline = (
    baseline_comparison_metrics.sort_values("rmsle", ascending=True).head(1).copy()
)

best_baseline_model = best_baseline["model"].iloc[0]
best_baseline_rmsle = best_baseline["rmsle"].iloc[0]
best_baseline_mae = best_baseline["mae"].iloc[0]
best_baseline_wape = best_baseline["wape"].iloc[0]
best_baseline_total_bias_pct = best_baseline["total_bias_pct"].iloc[0]
best_baseline_abs_total_bias_pct = abs(best_baseline_total_bias_pct)

lightgbm_row = lightgbm_metrics_for_comparison.iloc[0]

lightgbm_vs_best_baseline = pd.DataFrame(
    [
        {
            "model": MODEL_VERSION,
            "benchmark_model": best_baseline_model,
            "rmsle": lightgbm_row["rmsle"],
            "benchmark_rmsle": best_baseline_rmsle,
            "rmsle_improvement_abs": best_baseline_rmsle - lightgbm_row["rmsle"],
            "rmsle_improvement_pct": (
                (best_baseline_rmsle - lightgbm_row["rmsle"]) / best_baseline_rmsle
            ),
            "mae": lightgbm_row["mae"],
            "benchmark_mae": best_baseline_mae,
            "mae_improvement_abs": best_baseline_mae - lightgbm_row["mae"],
            "mae_improvement_pct": (
                (best_baseline_mae - lightgbm_row["mae"]) / best_baseline_mae
            ),
            "wape": lightgbm_row["wape"],
            "benchmark_wape": best_baseline_wape,
            "wape_improvement_abs": best_baseline_wape - lightgbm_row["wape"],
            "wape_improvement_pct": (
                (best_baseline_wape - lightgbm_row["wape"]) / best_baseline_wape
            ),
            "total_bias_pct": lightgbm_row["total_bias_pct"],
            "benchmark_total_bias_pct": best_baseline_total_bias_pct,
            "abs_total_bias_pct": abs(lightgbm_row["total_bias_pct"]),
            "benchmark_abs_total_bias_pct": best_baseline_abs_total_bias_pct,
        }
    ]
)

model_comparison_display = model_comparison.copy()

percentage_columns = [
    "wape",
    "total_bias_pct",
    "abs_total_bias_pct",
]

for column in percentage_columns:
    model_comparison_display[column] = model_comparison_display[column] * 100

lightgbm_vs_best_baseline_display = lightgbm_vs_best_baseline.copy()

percentage_columns_comparison = [
    "rmsle_improvement_pct",
    "mae_improvement_pct",
    "wape",
    "benchmark_wape",
    "wape_improvement_pct",
    "total_bias_pct",
    "benchmark_total_bias_pct",
    "abs_total_bias_pct",
    "benchmark_abs_total_bias_pct",
]

for column in percentage_columns_comparison:
    lightgbm_vs_best_baseline_display[column] = (
        lightgbm_vs_best_baseline_display[column] * 100
    )

display(model_comparison_display)
display(lightgbm_vs_best_baseline_display)


,model,rows,rmsle,mae,wape,bias,total_bias_pct,actual_total_sales,predicted_total_sales,model_type,abs_bias,abs_total_bias_pct,rmsle_rank
0,lightgbm_v1,28512,0.4167,68.1379,14.5861,-0.3896,-0.0834,"13,319,179.7816","13,308,071.3331",machine_learning,0.3896,0.0834,1
1,baseline_rolling_mean_28_origin,28512,0.5216,97.2231,20.8123,6.2760,1.3435,"13,319,179.7816","13,498,121.0000",baseline,6.2760,1.3435,2
2,baseline_last_7_day_pattern,28512,0.6170,96.5354,20.6651,1.9146,0.4099,"13,319,179.7816","13,373,770.0000",baseline,1.9146,0.4099,3
3,baseline_last_observed,28512,0.6595,207.5058,44.4202,163.4701,34.9936,"13,319,179.7816","17,980,040.0000",baseline,163.4701,34.9936,4
4,baseline_store_family_weekday_avg,28512,0.6916,142.7926,30.5672,-113.5773,-24.3132,"13,319,179.7816","10,080,863.0000",baseline,113.5773,24.3132,5
5,baseline_zero,28512,4.4195,467.1429,100.0000,-467.1429,-100.0000,"13,319,179.7816",0.0000,baseline,467.1429,100.0000,6


,model,benchmark_model,rmsle,benchmark_rmsle,rmsle_improvement_abs,rmsle_improvement_pct,mae,benchmark_mae,mae_improvement_abs,mae_improvement_pct,wape,benchmark_wape,wape_improvement_abs,wape_improvement_pct,total_bias_pct,benchmark_total_bias_pct,abs_total_bias_pct,benchmark_abs_total_bias_pct
0,lightgbm_v1,baseline_rolling_mean_28_origin,0.4167,0.5216,0.1049,20.1086,68.1379,97.2231,29.0851,29.9159,14.5861,20.8123,0.0623,29.9159,-0.0834,1.3435,0.0834,1.3435


### Section conclusion

LightGBM v1 clearly outperforms the best baseline model.

Compared with `baseline_rolling_mean_28_origin`, LightGBM v1 reduces RMSLE from 0.5216 to 0.4167, which represents an improvement of 20.11%.

The model also improves business-oriented metrics. MAE decreases from 97.22 to 68.14, and WAPE decreases from 20.81% to 14.59%. This means that LightGBM reduces the weighted absolute forecasting error by approximately 6.23 percentage points.

The total forecast bias also improves substantially. The best baseline overpredicted total sales by 1.34%, while LightGBM v1 has a nearly neutral total bias of -0.08%.

Based on this validation comparison, LightGBM v1 is accepted as the first main machine learning model candidate for the project.


# 12.Overfitting and Stability Review

This section reviews whether LightGBM v1 shows signs of excessive overfitting.

The model is evaluated by comparing training and validation RMSE on the log-transformed target, reviewing the best iteration, and analyzing the learning curve.

A moderate train-validation gap is expected, but the model should not show a large divergence between training and validation performance.


In [9]:
learning_curve = pd.DataFrame(
    {
        "iteration": np.arange(1, len(evals_result["train"]["rmse"]) + 1),
        "train_rmse_log": evals_result["train"]["rmse"],
        "valid_rmse_log": evals_result["valid"]["rmse"],
    }
)

learning_curve["rmse_log_gap"] = (
    learning_curve["valid_rmse_log"] - learning_curve["train_rmse_log"]
)

learning_curve["rmse_log_gap_pct_vs_train"] = (
    learning_curve["rmse_log_gap"] / learning_curve["train_rmse_log"]
)

best_iteration_row = learning_curve.loc[
    learning_curve["valid_rmse_log"].idxmin()
].copy()

final_iteration_row = learning_curve.tail(1).iloc[0].copy()

overfitting_summary = pd.DataFrame(
    [
        {
            "model": MODEL_VERSION,
            "total_iterations_completed": len(learning_curve),
            "best_iteration": int(best_iteration_row["iteration"]),
            "best_train_rmse_log": best_iteration_row["train_rmse_log"],
            "best_valid_rmse_log": best_iteration_row["valid_rmse_log"],
            "best_rmse_log_gap": best_iteration_row["rmse_log_gap"],
            "best_rmse_log_gap_pct_vs_train": best_iteration_row[
                "rmse_log_gap_pct_vs_train"
            ],
            "final_iteration": int(final_iteration_row["iteration"]),
            "final_train_rmse_log": final_iteration_row["train_rmse_log"],
            "final_valid_rmse_log": final_iteration_row["valid_rmse_log"],
            "valid_rmse_change_best_to_final": (
                final_iteration_row["valid_rmse_log"]
                - best_iteration_row["valid_rmse_log"]
            ),
            "early_stopping_triggered": len(learning_curve)
            < FULL_TRAINING_BOOST_ROUNDS,
        }
    ]
)

learning_curve_checkpoints = learning_curve[
    (learning_curve["iteration"] % 50 == 0)
    | (learning_curve["iteration"] == best_iteration)
    | (learning_curve["iteration"] == len(learning_curve))
].copy()

display(overfitting_summary)
display(learning_curve_checkpoints.tail(15))


,model,total_iterations_completed,best_iteration,best_train_rmse_log,best_valid_rmse_log,best_rmse_log_gap,best_rmse_log_gap_pct_vs_train,final_iteration,final_train_rmse_log,final_valid_rmse_log,valid_rmse_change_best_to_final,early_stopping_triggered
0,lightgbm_v1,1200,1174,0.3963,0.4167,0.0205,0.0516,1200,0.3954,0.4168,0.0000,False


,iteration,train_rmse_log,valid_rmse_log,rmse_log_gap,rmse_log_gap_pct_vs_train
549,550,0.4236,0.4253,0.0017,0.0040
599,600,0.4207,0.4245,0.0038,0.0089
649,650,0.4177,0.4238,0.0061,0.0146
699,700,0.4151,0.4230,0.0079,0.0192
749,750,0.4125,0.4228,0.0103,0.0251
799,800,0.4103,0.4228,0.0125,0.0304
849,850,0.4084,0.4215,0.0130,0.0319
899,900,0.4065,0.4206,0.0142,0.0349
949,950,0.4041,0.4196,0.0155,0.0384
999,1000,0.4021,0.4187,0.0165,0.0411


### Section conclusion

LightGBM v1 shows a controlled train-validation gap.

The best validation result is reached at iteration 1,174, with a training RMSE of 0.3963 and a validation RMSE of 0.4167 on the log-transformed target. The gap between train and validation is approximately 0.0205, equivalent to 5.16% of the training error.

The validation curve continues improving until late in the training process and does not degrade meaningfully by the final iteration. Early stopping was not triggered because the model reached the maximum number of boosting rounds.

This indicates that LightGBM v1 is learning useful general patterns without showing excessive overfitting. The model is stable enough for this first machine learning version, although future iterations could still explore stronger regularization, more validation windows, or controlled hyperparameter tuning.


# 13.Feature Importance Analysis

This section analyzes LightGBM feature importance.

Feature importance is used as a technical diagnostic to understand which variables contributed most to the model. It should not be interpreted as causal evidence.

The analysis includes both gain-based and split-based importance. Gain importance shows which features contributed most to reducing model error, while split importance shows how frequently each feature was used across the trees.


In [10]:
feature_importance = pd.DataFrame(
    {
        "feature": lightgbm_v1_model.feature_name(),
        "importance_gain": lightgbm_v1_model.feature_importance(importance_type="gain"),
        "importance_split": lightgbm_v1_model.feature_importance(
            importance_type="split"
        ),
    }
)

feature_importance["importance_gain_pct"] = (
    feature_importance["importance_gain"] / feature_importance["importance_gain"].sum()
)

feature_importance["importance_split_pct"] = (
    feature_importance["importance_split"]
    / feature_importance["importance_split"].sum()
)

feature_importance = feature_importance.sort_values(
    by="importance_gain",
    ascending=False,
).reset_index(drop=True)

feature_importance["gain_rank"] = (
    feature_importance["importance_gain"]
    .rank(method="min", ascending=False)
    .astype(int)
)

feature_importance["split_rank"] = (
    feature_importance["importance_split"]
    .rank(method="min", ascending=False)
    .astype(int)
)

top_feature_importance = feature_importance.head(25).copy()

feature_group_rules = {
    "store_product": [
        "store_nbr",
        "family",
        "city",
        "state",
        "store_type",
        "cluster",
    ],
    "promotion": [
        "onpromotion",
    ],
    "calendar": [
        "day_of_week",
        "day_of_month",
        "week_of_year",
        "month",
        "year",
        "is_weekend",
        "is_month_start",
        "is_month_end",
    ],
    "historical_sales": [
        "sales_lag_16",
        "sales_lag_21",
        "sales_lag_28",
        "sales_rolling_mean_7_shift_16",
        "sales_rolling_mean_14_shift_16",
        "sales_rolling_mean_28_shift_16",
    ],
    "oil": [
        "dcoilwtico",
        "oil_row_exists_in_raw",
        "dcoilwtico_missing_before_imputation",
        "dcoilwtico_was_imputed",
    ],
    "events": [
        "calendar_event_count",
        "transferred_event_count",
        "unique_locale_names",
        "non_transferred_event_count",
        "is_calendar_event",
        "has_non_transferred_event",
        "has_transferred_event",
    ],
    "holiday_encoded": [
        feature
        for feature in model_features
        if feature.startswith("holiday_type_") or feature.startswith("holiday_locale_")
    ],
}


def assign_feature_group(feature):
    """Assign a feature to a high-level business/technical group."""
    for group_name, group_features in feature_group_rules.items():
        if feature in group_features:
            return group_name

    return "other"


feature_importance["feature_group"] = feature_importance["feature"].apply(
    assign_feature_group
)

feature_group_importance = feature_importance.groupby(
    "feature_group", as_index=False
).agg(
    features=("feature", "count"),
    importance_gain=("importance_gain", "sum"),
    importance_split=("importance_split", "sum"),
)

feature_group_importance["importance_gain_pct"] = (
    feature_group_importance["importance_gain"]
    / feature_group_importance["importance_gain"].sum()
)

feature_group_importance["importance_split_pct"] = (
    feature_group_importance["importance_split"]
    / feature_group_importance["importance_split"].sum()
)

feature_group_importance = feature_group_importance.sort_values(
    by="importance_gain",
    ascending=False,
).reset_index(drop=True)

display(top_feature_importance)
display(feature_group_importance)


,feature,importance_gain,importance_split,importance_gain_pct,importance_split_pct,gain_rank,split_rank
0,sales_rolling_mean_7_shift_16,"101,848,059.5006",1511,0.5872,0.0429,1,8
1,sales_lag_21,"43,542,338.4180",1395,0.2511,0.0396,2,9
2,sales_rolling_mean_28_shift_16,"7,869,209.8913",1862,0.0454,0.0529,3,5
3,sales_lag_16,"5,891,257.0661",1187,0.0340,0.0337,4,12
4,family,"3,572,612.4309",6439,0.0206,0.1828,5,1
5,sales_lag_28,"2,806,113.4219",1254,0.0162,0.0356,6,10
6,dcoilwtico,"1,186,312.8493",2241,0.0068,0.0636,7,4
7,month,"1,137,158.3498",3051,0.0066,0.0866,8,3
8,onpromotion,"949,232.8999",818,0.0055,0.0232,9,15
9,sales_rolling_mean_14_shift_16,"784,034.5732",948,0.0045,0.0269,10,14


,feature_group,features,importance_gain,importance_split,importance_gain_pct,importance_split_pct
0,historical_sales,6,"162,741,012.8712",8157,0.9383,0.2316
1,store_product,6,"4,539,306.7801",13775,0.0262,0.3911
2,calendar,8,"3,438,345.6210",9193,0.0198,0.2610
3,oil,4,"1,246,225.5722",2446,0.0072,0.0694
4,promotion,1,"949,232.8999",818,0.0055,0.0232
5,holiday_encoded,9,"366,161.6756",606,0.0021,0.0172
6,events,7,"156,843.6739",225,0.0009,0.0064


### Section conclusion

The LightGBM v1 feature importance is technically coherent for a forecasting model.

Historical sales features dominate the model, representing 93.83% of total gain importance. The most important individual feature is `sales_rolling_mean_7_shift_16`, followed by `sales_lag_21`, `sales_rolling_mean_28_shift_16`, and `sales_lag_16`.

This confirms that the model is mainly learning from recent safe historical demand patterns. This is expected and acceptable because these lag and rolling features were shifted to avoid future information leakage.

Store-product structure, calendar variables, promotions, oil, and holiday/event indicators also contribute to the model, although with lower gain importance. These variables help LightGBM adjust the historical demand signal according to business context.

The importance analysis does not provide causal interpretation, but it supports that LightGBM v1 is relying on reasonable and business-relevant predictors.


# 14.Error Analysis by Business Segment

This section analyzes LightGBM v1 validation errors across relevant business segments.

The objective is to understand where the model performs well and where forecasting risk remains high.

The analysis covers:

* validation date;
* forecast horizon day;
* store;
* product family;
* promotion status;
* store-family combinations.

This helps translate model performance into replenishment and planning implications.


In [11]:
def evaluate_forecast_group(group, group_columns):
    """
    Compute forecasting metrics for a grouped validation prediction table.
    """
    y_true = group["sales"].to_numpy()
    y_pred = group["prediction"].to_numpy()

    return pd.Series(
        {
            "rows": len(group),
            "actual_total_sales": np.sum(y_true),
            "predicted_total_sales": np.sum(clip_forecast_predictions(y_pred)),
            "rmsle": rmsle_score(y_true, y_pred),
            "mae": mae_score(y_true, y_pred),
            "wape": wape_score(y_true, y_pred),
            "bias": bias_score(y_true, y_pred),
            "total_bias_pct": total_bias_pct_score(y_true, y_pred),
            "mean_absolute_error": group["absolute_error"].mean(),
            "total_absolute_error": group["absolute_error"].sum(),
        }
    )


def metrics_by_segment(prediction_data, group_columns):
    """
    Aggregate forecast metrics by one or more segment columns.
    """
    if isinstance(group_columns, str):
        group_columns = [group_columns]

    segment_metrics = (
        prediction_data.groupby(group_columns, observed=True)
        .apply(lambda group: evaluate_forecast_group(group, group_columns))
        .reset_index()
    )

    segment_metrics["error_contribution_pct"] = (
        segment_metrics["total_absolute_error"]
        / segment_metrics["total_absolute_error"].sum()
    )

    return segment_metrics


validation_dates = sorted(lightgbm_v1_validation_predictions["date"].unique())

date_to_horizon_day = {
    date: horizon_day for horizon_day, date in enumerate(validation_dates, start=1)
}

lightgbm_v1_validation_predictions["horizon_day"] = (
    lightgbm_v1_validation_predictions["date"].map(date_to_horizon_day).astype("int16")
)

lightgbm_v1_validation_predictions["promotion_status"] = np.where(
    lightgbm_v1_validation_predictions["onpromotion"] > 0,
    "with_promotion",
    "without_promotion",
)

metrics_by_date = metrics_by_segment(
    lightgbm_v1_validation_predictions,
    "date",
).sort_values("date")

metrics_by_horizon_day = metrics_by_segment(
    lightgbm_v1_validation_predictions,
    "horizon_day",
).sort_values("horizon_day")

metrics_by_store = metrics_by_segment(
    lightgbm_v1_validation_predictions,
    "store_nbr",
).sort_values("rmsle", ascending=False)

metrics_by_family = metrics_by_segment(
    lightgbm_v1_validation_predictions,
    "family",
).sort_values("rmsle", ascending=False)

metrics_by_promotion = metrics_by_segment(
    lightgbm_v1_validation_predictions,
    "promotion_status",
).sort_values("rmsle", ascending=False)

metrics_by_store_family = metrics_by_segment(
    lightgbm_v1_validation_predictions,
    ["store_nbr", "family"],
)

worst_dates_by_rmsle = metrics_by_date.sort_values(
    "rmsle",
    ascending=False,
).head(10)

worst_stores_by_rmsle = metrics_by_store.sort_values(
    "rmsle",
    ascending=False,
).head(10)

worst_families_by_rmsle = metrics_by_family.sort_values(
    "rmsle",
    ascending=False,
).head(10)

highest_error_store_family = metrics_by_store_family.sort_values(
    "total_absolute_error",
    ascending=False,
).head(20)

highest_wape_store_family = (
    metrics_by_store_family.loc[metrics_by_store_family["actual_total_sales"] > 0]
    .sort_values("wape", ascending=False)
    .head(20)
)

display(worst_dates_by_rmsle)
display(metrics_by_horizon_day)
display(worst_stores_by_rmsle)
display(worst_families_by_rmsle)
display(metrics_by_promotion)
display(highest_error_store_family)


,date,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct,mean_absolute_error,total_absolute_error,error_contribution_pct
14,2017-08-14,"1,782.0000","760,922.4061","747,618.3933",0.4587,58.3622,0.1367,-7.4658,-0.0175,58.3622,"104,001.4635",0.0535
15,2017-08-15,"1,782.0000","762,661.9359","713,556.6551",0.4568,63.4750,0.1483,-27.5563,-0.0644,63.4750,"113,112.4911",0.0582
12,2017-08-12,"1,782.0000","792,630.5351","902,083.8085",0.4346,110.5723,0.2486,61.4216,0.1381,110.5723,"197,039.9038",0.1014
13,2017-08-13,"1,782.0000","865,639.6775","968,631.1618",0.4279,95.5015,0.1966,57.7954,0.1190,95.5015,"170,183.6859",0.0876
8,2017-08-08,"1,782.0000","717,766.3491","701,573.8697",0.4221,49.5954,0.1231,-9.0867,-0.0226,49.5954,"88,379.0178",0.0455
11,2017-08-11,"1,782.0000","826,373.7220","875,394.0445",0.4208,90.9157,0.1961,27.5086,0.0593,90.9157,"162,011.7207",0.0834
4,2017-08-04,"1,782.0000","827,775.6861","821,862.0358",0.4183,63.9099,0.1376,-3.3185,-0.0071,63.9099,"113,887.5088",0.0586
7,2017-08-07,"1,782.0000","797,464.9638","775,794.7373",0.4152,52.7765,0.1179,-12.1606,-0.0272,52.7765,"94,047.7854",0.0484
6,2017-08-06,"1,782.0000","1,049,559.1643","1,050,121.8972",0.4103,68.2812,0.1159,0.3158,0.0005,68.2812,"121,677.1222",0.0626
0,2017-07-31,"1,782.0000","885,856.8409","822,685.5227",0.4071,61.6388,0.1240,-35.4497,-0.0713,61.6388,"109,840.2592",0.0565


,horizon_day,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct,mean_absolute_error,total_absolute_error,error_contribution_pct
0,1,"1,782.0000","885,856.8409","822,685.5227",0.4071,61.6388,0.1240,-35.4497,-0.0713,61.6388,"109,840.2592",0.0565
1,2,"1,782.0000","988,527.7632","932,465.5620",0.3999,79.5776,0.1435,-31.4603,-0.0567,79.5776,"141,807.3573",0.0730
2,3,"1,782.0000","964,712.0161","900,616.5249",0.3987,66.3372,0.1225,-35.9683,-0.0664,66.3372,"118,212.9763",0.0608
3,4,"1,782.0000","728,068.4851","738,391.6381",0.4006,58.0769,0.1421,5.7930,0.0142,58.0769,"103,492.9692",0.0533
4,5,"1,782.0000","827,775.6861","821,862.0358",0.4183,63.9099,0.1376,-3.3185,-0.0071,63.9099,"113,887.5088",0.0586
5,6,"1,782.0000","965,693.6505","976,254.0942",0.3856,67.2966,0.1242,5.9262,0.0109,67.2966,"119,922.6292",0.0617
6,7,"1,782.0000","1,049,559.1643","1,050,121.8972",0.4103,68.2812,0.1159,0.3158,0.0005,68.2812,"121,677.1222",0.0626
7,8,"1,782.0000","797,464.9638","775,794.7373",0.4152,52.7765,0.1179,-12.1606,-0.0272,52.7765,"94,047.7854",0.0484
8,9,"1,782.0000","717,766.3491","701,573.8697",0.4221,49.5954,0.1231,-9.0867,-0.0226,49.5954,"88,379.0178",0.0455
9,10,"1,782.0000","734,139.6740","731,109.3428",0.3984,48.7019,0.1182,-1.7005,-0.0041,48.7019,"86,786.7565",0.0447


,store_nbr,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct,mean_absolute_error,total_absolute_error,error_contribution_pct
17,18,528.0000,"162,414.1110","74,287.9679",0.7795,171.6052,0.5579,-166.9056,-0.5426,171.6052,"90,607.5217",0.0466
24,25,528.0000,"154,921.5371","102,871.9019",0.5362,100.9023,0.3439,-98.5789,-0.3360,100.9023,"53,276.4267",0.0274
18,19,528.0000,"150,509.4550","147,791.0897",0.4895,37.5780,0.1318,-5.1484,-0.0181,37.5780,"19,841.1912",0.0102
25,26,528.0000,"74,602.8770","72,859.6925",0.4804,31.0531,0.2198,-3.3015,-0.0234,31.0531,"16,396.0530",0.0084
21,22,528.0000,"121,737.9160","110,777.9023",0.4781,49.5535,0.2149,-20.7576,-0.0900,49.5535,"26,164.2456",0.0135
49,50,528.0000,"326,493.2400","346,519.2915",0.4716,87.3685,0.1413,37.9281,0.0613,87.3685,"46,130.5630",0.0237
13,14,528.0000,"136,648.1410","111,743.1643",0.4617,58.6486,0.2266,-47.1685,-0.1823,58.6486,"30,966.4414",0.0159
46,47,528.0000,"581,797.1710","599,958.0826",0.4511,130.3536,0.1183,34.3957,0.0312,130.3536,"68,826.7234",0.0354
47,48,528.0000,"403,740.9601","400,043.2456",0.4439,95.0741,0.1243,-7.0032,-0.0092,95.0741,"50,199.1361",0.0258
35,36,528.0000,"213,441.4689","182,970.3383",0.4432,78.1246,0.1933,-57.7105,-0.1428,78.1246,"41,249.8011",0.0212


,family,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct,mean_absolute_error,total_absolute_error,error_contribution_pct
31,SCHOOL AND OFFICE SUPPLIES,864.0000,"51,795.0000","12,979.7379",0.7613,45.8727,0.7652,-44.9251,-0.7494,45.8727,"39,634.0251",0.0204
21,LINGERIE,864.0000,"6,716.0000","5,583.7487",0.6256,3.8798,0.4991,-1.3105,-0.1686,3.8798,"3,352.1457",0.0017
13,GROCERY II,864.0000,"25,031.0000","24,875.2574",0.5995,10.1322,0.3497,-0.1803,-0.0062,10.1322,"8,754.2632",0.0045
6,CELEBRATION,864.0000,"10,768.0000","9,661.8119",0.5584,4.9505,0.3972,-1.2803,-0.1027,4.9505,"4,277.2728",0.0022
23,MAGAZINES,864.0000,"6,146.0000","4,390.4965",0.5355,2.9837,0.4194,-2.0318,-0.2856,2.9837,"2,577.9242",0.0013
14,HARDWARE,864.0000,"1,252.0000","1,016.1246",0.5311,1.0663,0.7359,-0.2730,-0.1884,1.0663,921.3263,0.0005
19,LADIESWEAR,864.0000,"8,711.0000","8,982.3343",0.5214,3.7725,0.3742,0.3140,0.0311,3.7725,"3,259.3997",0.0017
0,AUTOMOTIVE,864.0000,"6,381.0000","5,551.6595",0.5093,2.8386,0.3844,-0.9599,-0.1300,2.8386,"2,452.5590",0.0013
32,SEAFOOD,864.0000,"17,260.9190","16,597.1100",0.5074,4.4647,0.2235,-0.7683,-0.0385,4.4647,"3,857.5122",0.0020
15,HOME AND KITCHEN I,864.0000,"27,020.0000","21,517.0848",0.4939,11.1352,0.3561,-6.3691,-0.2037,11.1352,"9,620.8276",0.0050


,promotion_status,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct,mean_absolute_error,total_absolute_error,error_contribution_pct
1,without_promotion,"16,031.0000","883,294.2805","847,387.4467",0.4592,9.6061,0.1743,-2.2398,-0.0407,9.6061,"153,994.8810",0.0793
0,with_promotion,"12,481.0000","12,435,885.5011","12,460,683.8864",0.3547,143.3182,0.1438,1.9869,0.0020,143.3182,"1,788,754.3467",0.9207


,store_nbr,family,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct,mean_absolute_error,total_absolute_error,error_contribution_pct
573,18,GROCERY I,16.0000,"47,553.6490","17,801.8818",1.1772,"1,861.4414",0.6263,"-1,859.4855",-0.6256,"1,861.4414","29,783.0621",0.0153
564,18,BEVERAGES,16.0000,"34,551.0000","13,799.5540",1.1665,"1,352.1593",0.6262,"-1,296.9654",-0.6006,"1,352.1593","21,634.5491",0.0111
903,28,GROCERY I,16.0000,"92,939.8820","74,431.1081",0.2824,"1,287.9797",0.2217,"-1,156.7984",-0.1991,"1,287.9797","20,607.6747",0.0106
1224,38,BEVERAGES,16.0000,"42,227.0000","47,214.9216",0.4872,"1,284.4387",0.4867,311.7451,0.1181,"1,284.4387","20,551.0188",0.0106
1752,54,BEVERAGES,16.0000,"62,539.0000","51,368.3679",0.3987,"1,218.4616",0.3117,-698.1645,-0.1786,"1,218.4616","19,495.3852",0.0100
639,20,GROCERY I,16.0000,"103,970.5650","86,640.8496",0.2226,"1,155.2309",0.1778,"-1,083.1072",-0.1667,"1,155.2309","18,483.6946",0.0095
1299,40,GROCERY I,16.0000,"102,278.0000","84,707.7187",0.2478,"1,140.7356",0.1785,"-1,098.1426",-0.1718,"1,140.7356","18,251.7692",0.0094
1158,36,BEVERAGES,16.0000,"64,111.0000","45,914.2877",0.3602,"1,137.2945",0.2838,"-1,137.2945",-0.2838,"1,137.2945","18,196.7123",0.0094
1290,40,BEVERAGES,16.0000,"85,249.0000","70,873.3025",0.3046,"1,129.4591",0.2120,-898.4811,-0.1686,"1,129.4591","18,071.3456",0.0093
69,3,BEVERAGES,16.0000,"126,131.0000","140,123.7708",0.1586,"1,072.1468",0.1360,874.5482,0.1109,"1,072.1468","17,154.3482",0.0088


### Section conclusion

The segment-level error analysis shows that LightGBM v1 performs well globally, but forecasting risk remains concentrated in specific dates, stores, families, and high-volume store-family combinations.

The largest date-level errors appear toward the end of the validation horizon, especially between horizon days 13 and 16. Horizon days 13 and 14 show the highest WAPE and strong overprediction, while days 15 and 16 show the highest RMSLE values.

At store level, store 18 is the clearest operational risk. It has the highest RMSLE, a WAPE above 55%, and a strong underprediction bias. Stores 25, 14, 36, 47, 48, and 50 also require monitoring, although their risk profile differs by volume and bias direction.

At family level, `SCHOOL AND OFFICE SUPPLIES` remains the most difficult category. LightGBM strongly underpredicts this family, suggesting that the current feature set does not fully capture its seasonal or event-driven demand pattern. However, its contribution to total absolute error is limited compared with high-volume categories.

Promotion rows perform better than non-promotion rows in relative metrics, but they account for more than 90% of total absolute error because they concentrate most validation sales volume. This means promotion periods remain the most important operational area for forecast review.

The highest absolute errors are concentrated in high-volume store-family combinations, especially `GROCERY I`, `BEVERAGES`, and `PRODUCE`. From a replenishment perspective, these combinations are more important than low-volume categories with high relative errors.


# 15.Segment Comparison Against the Best Baseline

This section compares LightGBM v1 against the best baseline model at segment level.

The objective is to identify where LightGBM improves the previous benchmark and where forecasting errors remain difficult.

This comparison helps distinguish between:

* segments where LightGBM provides clear operational improvement;
* segments where both models still struggle;
* business areas that may require additional features or model iterations.


In [12]:
best_baseline_model = lightgbm_vs_best_baseline["benchmark_model"].iloc[0]

baseline_best_validation_predictions = baseline_validation_predictions.copy()

if "model" in baseline_best_validation_predictions.columns:
    available_baseline_models = (
        baseline_best_validation_predictions["model"].dropna().unique()
    )

    if best_baseline_model in available_baseline_models:
        baseline_best_validation_predictions = baseline_best_validation_predictions.loc[
            baseline_best_validation_predictions["model"] == best_baseline_model
        ].copy()

prediction_column_candidates = [
    "prediction",
    "predicted_sales",
    "baseline_prediction",
    "forecast",
    best_baseline_model,
    f"{best_baseline_model}_prediction",
]

available_prediction_columns = [
    column
    for column in prediction_column_candidates
    if column in baseline_best_validation_predictions.columns
]

if not available_prediction_columns:
    raise ValueError(
        "No prediction column found in baseline_validation_predictions. "
        f"Available columns: {list(baseline_best_validation_predictions.columns)}"
    )

baseline_prediction_column = available_prediction_columns[0]

baseline_best_validation_predictions = baseline_best_validation_predictions.rename(
    columns={baseline_prediction_column: "prediction"}
)

required_baseline_prediction_columns = [
    "date",
    "store_nbr",
    "family",
    "sales",
    "prediction",
]

missing_baseline_prediction_columns = [
    column
    for column in required_baseline_prediction_columns
    if column not in baseline_best_validation_predictions.columns
]

if missing_baseline_prediction_columns:
    raise ValueError(
        "Missing required columns in baseline validation predictions: "
        f"{missing_baseline_prediction_columns}. "
        f"Available columns: {list(baseline_best_validation_predictions.columns)}"
    )

if "onpromotion" not in baseline_best_validation_predictions.columns:
    baseline_best_validation_predictions = baseline_best_validation_predictions.merge(
        valid_features[["date", "store_nbr", "family", "onpromotion"]],
        on=["date", "store_nbr", "family"],
        how="left",
    )

baseline_best_validation_predictions["date"] = pd.to_datetime(
    baseline_best_validation_predictions["date"]
)

baseline_best_validation_predictions["prediction"] = clip_forecast_predictions(
    baseline_best_validation_predictions["prediction"]
)

baseline_best_validation_predictions["residual"] = (
    baseline_best_validation_predictions["prediction"]
    - baseline_best_validation_predictions["sales"]
)

baseline_best_validation_predictions["absolute_error"] = (
    baseline_best_validation_predictions["residual"].abs()
)

baseline_best_validation_predictions["horizon_day"] = (
    baseline_best_validation_predictions["date"]
    .map(date_to_horizon_day)
    .astype("int16")
)

baseline_best_validation_predictions["promotion_status"] = np.where(
    baseline_best_validation_predictions["onpromotion"] > 0,
    "with_promotion",
    "without_promotion",
)


def compare_segment_metrics(
    lightgbm_segment_metrics, baseline_segment_metrics, segment_columns
):
    """
    Compare LightGBM segment metrics against baseline segment metrics.
    """
    if isinstance(segment_columns, str):
        segment_columns = [segment_columns]

    metric_columns = [
        "rows",
        "actual_total_sales",
        "predicted_total_sales",
        "rmsle",
        "mae",
        "wape",
        "bias",
        "total_bias_pct",
        "total_absolute_error",
        "error_contribution_pct",
    ]

    lightgbm_subset = lightgbm_segment_metrics[segment_columns + metric_columns].copy()

    baseline_subset = baseline_segment_metrics[segment_columns + metric_columns].copy()

    comparison = lightgbm_subset.merge(
        baseline_subset,
        on=segment_columns,
        how="inner",
        suffixes=("_lightgbm", "_baseline"),
    )

    comparison["rmsle_improvement_abs"] = (
        comparison["rmsle_baseline"] - comparison["rmsle_lightgbm"]
    )

    comparison["rmsle_improvement_pct"] = (
        comparison["rmsle_improvement_abs"] / comparison["rmsle_baseline"]
    )

    comparison["mae_improvement_abs"] = (
        comparison["mae_baseline"] - comparison["mae_lightgbm"]
    )

    comparison["mae_improvement_pct"] = (
        comparison["mae_improvement_abs"] / comparison["mae_baseline"]
    )

    comparison["wape_improvement_pp"] = (
        comparison["wape_baseline"] - comparison["wape_lightgbm"]
    ) * 100

    comparison["absolute_error_reduction"] = (
        comparison["total_absolute_error_baseline"]
        - comparison["total_absolute_error_lightgbm"]
    )

    comparison["absolute_error_reduction_pct"] = (
        comparison["absolute_error_reduction"]
        / comparison["total_absolute_error_baseline"]
    )

    comparison["abs_total_bias_pct_lightgbm"] = comparison[
        "total_bias_pct_lightgbm"
    ].abs()

    comparison["abs_total_bias_pct_baseline"] = comparison[
        "total_bias_pct_baseline"
    ].abs()

    comparison["abs_total_bias_improvement_pp"] = (
        comparison["abs_total_bias_pct_baseline"]
        - comparison["abs_total_bias_pct_lightgbm"]
    ) * 100

    return comparison


baseline_metrics_by_date = metrics_by_segment(
    baseline_best_validation_predictions,
    "date",
).sort_values("date")

baseline_metrics_by_horizon_day = metrics_by_segment(
    baseline_best_validation_predictions,
    "horizon_day",
).sort_values("horizon_day")

baseline_metrics_by_store = metrics_by_segment(
    baseline_best_validation_predictions,
    "store_nbr",
)

baseline_metrics_by_family = metrics_by_segment(
    baseline_best_validation_predictions,
    "family",
)

baseline_metrics_by_promotion = metrics_by_segment(
    baseline_best_validation_predictions,
    "promotion_status",
)

segment_comparison_by_date = compare_segment_metrics(
    metrics_by_date,
    baseline_metrics_by_date,
    "date",
)

segment_comparison_by_horizon_day = compare_segment_metrics(
    metrics_by_horizon_day,
    baseline_metrics_by_horizon_day,
    "horizon_day",
)

segment_comparison_by_store = compare_segment_metrics(
    metrics_by_store,
    baseline_metrics_by_store,
    "store_nbr",
)

segment_comparison_by_family = compare_segment_metrics(
    metrics_by_family,
    baseline_metrics_by_family,
    "family",
)

segment_comparison_by_promotion = compare_segment_metrics(
    metrics_by_promotion,
    baseline_metrics_by_promotion,
    "promotion_status",
)

date_comparison_display = segment_comparison_by_date[
    [
        "date",
        "rmsle_lightgbm",
        "rmsle_baseline",
        "rmsle_improvement_pct",
        "wape_lightgbm",
        "wape_baseline",
        "wape_improvement_pp",
        "total_bias_pct_lightgbm",
        "total_bias_pct_baseline",
    ]
].sort_values("date")

horizon_comparison_display = segment_comparison_by_horizon_day[
    [
        "horizon_day",
        "rmsle_lightgbm",
        "rmsle_baseline",
        "rmsle_improvement_pct",
        "wape_lightgbm",
        "wape_baseline",
        "wape_improvement_pp",
        "total_bias_pct_lightgbm",
        "total_bias_pct_baseline",
    ]
].sort_values("horizon_day")

promotion_comparison_display = segment_comparison_by_promotion[
    [
        "promotion_status",
        "rmsle_lightgbm",
        "rmsle_baseline",
        "rmsle_improvement_pct",
        "wape_lightgbm",
        "wape_baseline",
        "wape_improvement_pp",
        "total_bias_pct_lightgbm",
        "total_bias_pct_baseline",
        "absolute_error_reduction_pct",
    ]
].sort_values("promotion_status")

store_comparison_worst_remaining = segment_comparison_by_store.sort_values(
    "rmsle_lightgbm",
    ascending=False,
).head(10)

family_comparison_worst_remaining = segment_comparison_by_family.sort_values(
    "rmsle_lightgbm",
    ascending=False,
).head(10)

store_comparison_largest_absolute_error_reduction = (
    segment_comparison_by_store.sort_values(
        "absolute_error_reduction", ascending=False
    ).head(10)
)

family_comparison_largest_absolute_error_reduction = (
    segment_comparison_by_family.sort_values(
        "absolute_error_reduction", ascending=False
    ).head(10)
)

display(date_comparison_display)
display(horizon_comparison_display)
display(promotion_comparison_display)
display(store_comparison_worst_remaining)
display(family_comparison_worst_remaining)
display(store_comparison_largest_absolute_error_reduction)
display(family_comparison_largest_absolute_error_reduction)


,date,rmsle_lightgbm,rmsle_baseline,rmsle_improvement_pct,wape_lightgbm,wape_baseline,wape_improvement_pp,total_bias_pct_lightgbm,total_bias_pct_baseline
0,2017-07-31,0.4071,0.4724,0.1382,0.1240,0.1372,1.3254,-0.0713,-0.0477
1,2017-08-01,0.3999,0.4671,0.1438,0.1435,0.1916,4.8120,-0.0567,-0.1466
2,2017-08-02,0.3987,0.4742,0.1591,0.1225,0.1765,5.4010,-0.0664,-0.1255
3,2017-08-03,0.4006,0.5054,0.2074,0.1421,0.2473,10.5159,0.0142,0.1587
4,2017-08-04,0.4183,0.5123,0.1835,0.1376,0.1783,4.0752,-0.0071,0.0192
5,2017-08-05,0.3856,0.4943,0.2198,0.1242,0.1723,4.8079,0.0109,-0.1264
6,2017-08-06,0.4103,0.5430,0.2445,0.1159,0.2495,13.3586,0.0005,-0.1962
7,2017-08-07,0.4152,0.5247,0.2088,0.1179,0.1563,3.8394,-0.0272,0.0579
8,2017-08-08,0.4221,0.5436,0.2235,0.1231,0.2774,15.4252,-0.0226,0.1754
9,2017-08-09,0.3984,0.5216,0.2363,0.1182,0.2503,13.2098,-0.0041,0.1491


,horizon_day,rmsle_lightgbm,rmsle_baseline,rmsle_improvement_pct,wape_lightgbm,wape_baseline,wape_improvement_pp,total_bias_pct_lightgbm,total_bias_pct_baseline
0,1,0.4071,0.4724,0.1382,0.1240,0.1372,1.3254,-0.0713,-0.0477
1,2,0.3999,0.4671,0.1438,0.1435,0.1916,4.8120,-0.0567,-0.1466
2,3,0.3987,0.4742,0.1591,0.1225,0.1765,5.4010,-0.0664,-0.1255
3,4,0.4006,0.5054,0.2074,0.1421,0.2473,10.5159,0.0142,0.1587
4,5,0.4183,0.5123,0.1835,0.1376,0.1783,4.0752,-0.0071,0.0192
5,6,0.3856,0.4943,0.2198,0.1242,0.1723,4.8079,0.0109,-0.1264
6,7,0.4103,0.5430,0.2445,0.1159,0.2495,13.3586,0.0005,-0.1962
7,8,0.4152,0.5247,0.2088,0.1179,0.1563,3.8394,-0.0272,0.0579
8,9,0.4221,0.5436,0.2235,0.1231,0.2774,15.4252,-0.0226,0.1754
9,10,0.3984,0.5216,0.2363,0.1182,0.2503,13.2098,-0.0041,0.1491


,promotion_status,rmsle_lightgbm,rmsle_baseline,rmsle_improvement_pct,wape_lightgbm,wape_baseline,wape_improvement_pp,total_bias_pct_lightgbm,total_bias_pct_baseline,absolute_error_reduction_pct
1,with_promotion,0.3547,0.5583,0.3646,0.1438,0.2073,6.3490,0.0020,0.0095,0.3062
0,without_promotion,0.4592,0.4911,0.0650,0.1743,0.2193,4.4971,-0.0407,0.0693,0.2051


,store_nbr,rows_lightgbm,actual_total_sales_lightgbm,predicted_total_sales_lightgbm,rmsle_lightgbm,mae_lightgbm,wape_lightgbm,bias_lightgbm,total_bias_pct_lightgbm,total_absolute_error_lightgbm,error_contribution_pct_lightgbm,rows_baseline,actual_total_sales_baseline,predicted_total_sales_baseline,rmsle_baseline,mae_baseline,wape_baseline,bias_baseline,total_bias_pct_baseline,total_absolute_error_baseline,error_contribution_pct_baseline,rmsle_improvement_abs,rmsle_improvement_pct,mae_improvement_abs,mae_improvement_pct,wape_improvement_pp,absolute_error_reduction,absolute_error_reduction_pct,abs_total_bias_pct_lightgbm,abs_total_bias_pct_baseline,abs_total_bias_improvement_pp
0,18,528.0000,"162,414.1110","74,287.9679",0.7795,171.6052,0.5579,-166.9056,-0.5426,"90,607.5217",0.0466,528.0000,"162,414.1110","166,268.7188",0.5865,73.1475,0.2378,7.3004,0.0237,"38,621.8797",0.0139,-0.1930,-0.3291,-98.4577,-1.3460,-32.0081,"-51,985.6419",-1.3460,0.5426,0.0237,-51.8868
1,25,528.0000,"154,921.5371","102,871.9019",0.5362,100.9023,0.3439,-98.5789,-0.3360,"53,276.4267",0.0274,528.0000,"154,921.5371","125,560.6172",0.5120,81.7750,0.2787,-55.6078,-0.1895,"43,177.1875",0.0156,-0.0242,-0.0472,-19.1273,-0.2339,-6.5189,"-10,099.2392",-0.2339,0.3360,0.1895,-14.6453
2,19,528.0000,"150,509.4550","147,791.0897",0.4895,37.5780,0.1318,-5.1484,-0.0181,"19,841.1912",0.0102,528.0000,"150,509.4550","146,131.1250",0.6332,48.1923,0.1691,-8.2923,-0.0291,"25,445.5521",0.0092,0.1436,0.2268,10.6143,0.2202,3.7236,"5,604.3609",0.2202,0.0181,0.0291,1.1029
3,26,528.0000,"74,602.8770","72,859.6925",0.4804,31.0531,0.2198,-3.3015,-0.0234,"16,396.0530",0.0084,528.0000,"74,602.8770","80,547.8828",0.5709,47.9167,0.3391,11.2595,0.0797,"25,300.0240",0.0091,0.0905,0.1585,16.8636,0.3519,11.9352,"8,903.9710",0.3519,0.0234,0.0797,5.6323
4,22,528.0000,"121,737.9160","110,777.9023",0.4781,49.5535,0.2149,-20.7576,-0.0900,"26,164.2456",0.0135,528.0000,"121,737.9160","110,325.6719",0.5681,48.7713,0.2115,-21.6141,-0.0937,"25,751.2336",0.0093,0.0900,0.1584,-0.7822,-0.0160,-0.3393,-413.0121,-0.0160,0.0900,0.0937,0.3715
5,50,528.0000,"326,493.2400","346,519.2915",0.4716,87.3685,0.1413,37.9281,0.0613,"46,130.5630",0.0237,528.0000,"326,493.2400","346,601.5000",0.9147,116.4624,0.1883,38.0839,0.0616,"61,492.1648",0.0222,0.4431,0.4844,29.0939,0.2498,4.7050,"15,361.6019",0.2498,0.0613,0.0616,0.0252
6,14,528.0000,"136,648.1410","111,743.1643",0.4617,58.6486,0.2266,-47.1685,-0.1823,"30,966.4414",0.0159,528.0000,"136,648.1410","129,410.1953",0.5729,65.6186,0.2535,-13.7082,-0.0530,"34,646.5996",0.0125,0.1112,0.1942,6.9700,0.1062,2.6932,"3,680.1582",0.1062,0.1823,0.0530,-12.9288
7,47,528.0000,"581,797.1710","599,958.0826",0.4511,130.3536,0.1183,34.3957,0.0312,"68,826.7234",0.0354,528.0000,"581,797.1710","606,709.2500",0.8815,215.3197,0.1954,47.1819,0.0428,"113,688.7958",0.0410,0.4304,0.4883,84.9660,0.3946,7.7109,"44,862.0724",0.3946,0.0312,0.0428,1.1604
8,48,528.0000,"403,740.9601","400,043.2456",0.4439,95.0741,0.1243,-7.0032,-0.0092,"50,199.1361",0.0258,528.0000,"403,740.9601","402,595.8438",0.8574,189.5408,0.2479,-2.1687,-0.0028,"100,077.5509",0.0361,0.4135,0.4822,94.4667,0.4984,12.3541,"49,878.4148",0.4984,0.0092,0.0028,-0.6322
9,36,528.0000,"213,441.4689","182,970.3383",0.4432,78.1246,0.1933,-57.7105,-0.1428,"41,249.8011",0.0212,528.0000,"213,441.4689","179,143.8281",0.4820,94.0431,0.2326,-64.9577,-0.1607,"49,654.7343",0.0179,0.0388,0.0805,15.9184,0.1693,3.9378,"8,404.9332",0.1693,0.1428,0.1607,1.7928


,family,rows_lightgbm,actual_total_sales_lightgbm,predicted_total_sales_lightgbm,rmsle_lightgbm,mae_lightgbm,wape_lightgbm,bias_lightgbm,total_bias_pct_lightgbm,total_absolute_error_lightgbm,error_contribution_pct_lightgbm,rows_baseline,actual_total_sales_baseline,predicted_total_sales_baseline,rmsle_baseline,mae_baseline,wape_baseline,bias_baseline,total_bias_pct_baseline,total_absolute_error_baseline,error_contribution_pct_baseline,rmsle_improvement_abs,rmsle_improvement_pct,mae_improvement_abs,mae_improvement_pct,wape_improvement_pp,absolute_error_reduction,absolute_error_reduction_pct,abs_total_bias_pct_lightgbm,abs_total_bias_pct_baseline,abs_total_bias_improvement_pp
0,SCHOOL AND OFFICE SUPPLIES,864.0000,"51,795.0000","12,979.7379",0.7613,45.8727,0.7652,-44.9251,-0.7494,"39,634.0251",0.0204,864.0000,"51,795.0000","3,966.2856",1.6422,55.9101,0.9326,-55.3573,-0.9234,"48,306.3572",0.0174,0.8809,0.5364,10.0374,0.1795,16.7436,"8,672.3321",0.1795,0.7494,0.9234,17.4022
1,LINGERIE,864.0000,"6,716.0000","5,583.7487",0.6256,3.8798,0.4991,-1.3105,-0.1686,"3,352.1457",0.0017,864.0000,"6,716.0000","5,418.8574",0.6623,4.0846,0.5255,-1.5013,-0.1931,"3,529.0714",0.0013,0.0366,0.0553,0.2048,0.0501,2.6344,176.9258,0.0501,0.1686,0.1931,2.4552
2,GROCERY II,864.0000,"25,031.0000","24,875.2574",0.5995,10.1322,0.3497,-0.1803,-0.0062,"8,754.2632",0.0045,864.0000,"25,031.0000","29,641.7148",0.6795,12.6705,0.4373,5.3365,0.1842,"10,947.2857",0.0039,0.0800,0.1178,2.5382,0.2003,8.7612,"2,193.0224",0.2003,0.0062,0.1842,17.7978
3,CELEBRATION,864.0000,"10,768.0000","9,661.8119",0.5584,4.9505,0.3972,-1.2803,-0.1027,"4,277.2728",0.0022,864.0000,"10,768.0000","11,615.9990",0.6074,5.6066,0.4499,0.9815,0.0788,"4,844.0714",0.0017,0.0490,0.0806,0.6560,0.1170,5.2637,566.7986,0.1170,0.1027,0.0788,-2.3977
4,MAGAZINES,864.0000,"6,146.0000","4,390.4965",0.5355,2.9837,0.4194,-2.0318,-0.2856,"2,577.9242",0.0013,864.0000,"6,146.0000","5,551.4287",0.5441,2.9640,0.4167,-0.6882,-0.0967,"2,560.9286",0.0009,0.0086,0.0159,-0.0197,-0.0066,-0.2765,-16.9956,-0.0066,0.2856,0.0967,-18.8892
5,HARDWARE,864.0000,"1,252.0000","1,016.1246",0.5311,1.0663,0.7359,-0.2730,-0.1884,921.3263,0.0005,864.0000,"1,252.0000","1,272.5714",0.5557,1.1133,0.7683,0.0238,0.0164,961.8571,0.0003,0.0246,0.0442,0.0469,0.0421,3.2373,40.5308,0.0421,0.1884,0.0164,-17.1968
6,LADIESWEAR,864.0000,"8,711.0000","8,982.3343",0.5214,3.7725,0.3742,0.3140,0.0311,"3,259.3997",0.0017,864.0000,"8,711.0000","9,832.0000",0.5802,4.5050,0.4468,1.2975,0.1287,"3,892.3571",0.0014,0.0588,0.1013,0.7326,0.1626,7.2662,632.9575,0.1626,0.0311,0.1287,9.7539
7,AUTOMOTIVE,864.0000,"6,381.0000","5,551.6595",0.5093,2.8386,0.3844,-0.9599,-0.1300,"2,452.5590",0.0013,864.0000,"6,381.0000","6,098.8569",0.5385,3.0439,0.4121,-0.3266,-0.0442,"2,629.9286",0.0009,0.0292,0.0543,0.2053,0.0674,2.7797,177.3695,0.0674,0.1300,0.0442,-8.5754
8,SEAFOOD,864.0000,"17,260.9190","16,597.1100",0.5074,4.4647,0.2235,-0.7683,-0.0385,"3,857.5122",0.0020,864.0000,"17,260.9190","17,428.5723",0.5543,5.2606,0.2633,0.1940,0.0097,"4,545.1783",0.0016,0.0469,0.0847,0.7959,0.1513,3.9839,687.6661,0.1513,0.0385,0.0097,-2.8744
9,HOME AND KITCHEN I,864.0000,"27,020.0000","21,517.0848",0.4939,11.1352,0.3561,-6.3691,-0.2037,"9,620.8276",0.0050,864.0000,"27,020.0000","26,548.5703",0.5103,11.2262,0.3590,-0.5456,-0.0174,"9,699.4285",0.0035,0.0164,0.0321,0.0910,0.0081,0.2909,78.6009,0.0081,0.2037,0.0174,-18.6213


,store_nbr,rows_lightgbm,actual_total_sales_lightgbm,predicted_total_sales_lightgbm,rmsle_lightgbm,mae_lightgbm,wape_lightgbm,bias_lightgbm,total_bias_pct_lightgbm,total_absolute_error_lightgbm,error_contribution_pct_lightgbm,rows_baseline,actual_total_sales_baseline,predicted_total_sales_baseline,rmsle_baseline,mae_baseline,wape_baseline,bias_baseline,total_bias_pct_baseline,total_absolute_error_baseline,error_contribution_pct_baseline,rmsle_improvement_abs,rmsle_improvement_pct,mae_improvement_abs,mae_improvement_pct,wape_improvement_pp,absolute_error_reduction,absolute_error_reduction_pct,abs_total_bias_pct_lightgbm,abs_total_bias_pct_baseline,abs_total_bias_improvement_pp
25,44,528.0000,"656,651.3530","706,031.2357",0.4067,147.9411,0.1190,93.5225,0.0752,"78,112.9014",0.0402,528.0000,"656,651.3530","723,267.6250",0.7904,268.9446,0.2163,126.1671,0.1014,"142,002.7657",0.0512,0.3837,0.4854,121.0035,0.4499,9.7296,"63,889.8643",0.4499,0.0752,0.1014,2.6249
37,45,528.0000,"631,268.5195","649,615.6080",0.3763,121.7791,0.1019,34.7483,0.0291,"64,299.3528",0.0331,528.0000,"631,268.5195","657,554.5000",0.4696,229.4330,0.1919,49.7840,0.0416,"121,140.6054",0.0437,0.0934,0.1989,107.6539,0.4692,9.0043,"56,841.2526",0.4692,0.0291,0.0416,1.2576
40,46,528.0000,"456,761.5911","473,446.4362",0.3711,95.4528,0.1103,31.6001,0.0365,"50,399.0971",0.0259,528.0000,"456,761.5911","484,009.5000",0.4845,197.8869,0.2288,51.6059,0.0597,"104,484.2949",0.0377,0.1134,0.2341,102.4341,0.5176,11.8410,"54,085.1978",0.5176,0.0365,0.0597,2.3126
8,48,528.0000,"403,740.9601","400,043.2456",0.4439,95.0741,0.1243,-7.0032,-0.0092,"50,199.1361",0.0258,528.0000,"403,740.9601","402,595.8438",0.8574,189.5408,0.2479,-2.1687,-0.0028,"100,077.5509",0.0361,0.4135,0.4822,94.4667,0.4984,12.3541,"49,878.4148",0.4984,0.0092,0.0028,-0.6322
7,47,528.0000,"581,797.1710","599,958.0826",0.4511,130.3536,0.1183,34.3957,0.0312,"68,826.7234",0.0354,528.0000,"581,797.1710","606,709.2500",0.8815,215.3197,0.1954,47.1819,0.0428,"113,688.7958",0.0410,0.4304,0.4883,84.9660,0.3946,7.7109,"44,862.0724",0.3946,0.0312,0.0428,1.1604
44,51,528.0000,"364,628.8980","373,678.5440",0.3634,67.2989,0.0975,17.1395,0.0248,"35,533.7991",0.0183,528.0000,"364,628.8980","383,297.3750",0.4351,140.9010,0.2040,35.3569,0.0512,"74,395.7126",0.0268,0.0717,0.1648,73.6021,0.5224,10.6579,"38,861.9135",0.5224,0.0248,0.0512,2.6380
33,28,528.0000,"260,467.9650","240,695.4808",0.3898,85.8522,0.1740,-37.4479,-0.0759,"45,329.9564",0.0233,528.0000,"260,467.9650","232,912.2812",0.4793,141.4504,0.2867,-52.1888,-0.1058,"74,685.8047",0.0269,0.0895,0.1866,55.5982,0.3931,11.2704,"29,355.8483",0.3931,0.0759,0.1058,2.9882
48,27,528.0000,"236,019.9930","246,078.2452",0.3458,43.9039,0.0982,19.0497,0.0426,"23,181.2514",0.0119,528.0000,"236,019.9930","244,558.3438",0.4410,90.4865,0.2024,16.1711,0.0362,"47,776.8515",0.0172,0.0953,0.2160,46.5826,0.5148,10.4210,"24,595.6001",0.5148,0.0426,0.0362,-0.6440
18,21,528.0000,"235,963.3271","223,314.6740",0.4240,64.7179,0.1448,-23.9558,-0.0536,"34,171.0392",0.0176,528.0000,"235,963.3271","217,056.8281",0.4752,110.5713,0.2474,-35.8078,-0.0801,"58,381.6391",0.0211,0.0512,0.1078,45.8534,0.4147,10.2603,"24,210.5999",0.4147,0.0536,0.0801,2.6520
13,9,528.0000,"303,853.3809","294,036.2465",0.4335,69.3733,0.1205,-18.5931,-0.0323,"36,629.0952",0.0189,528.0000,"303,853.3809","276,694.9062",0.6902,114.3834,0.1988,-51.4364,-0.0894,"60,394.4354",0.0218,0.2566,0.3719,45.0101,0.3935,7.8213,"23,765.3402",0.3935,0.0323,0.0894,5.7071


,family,rows_lightgbm,actual_total_sales_lightgbm,predicted_total_sales_lightgbm,rmsle_lightgbm,mae_lightgbm,wape_lightgbm,bias_lightgbm,total_bias_pct_lightgbm,total_absolute_error_lightgbm,error_contribution_pct_lightgbm,rows_baseline,actual_total_sales_baseline,predicted_total_sales_baseline,rmsle_baseline,mae_baseline,wape_baseline,bias_baseline,total_bias_pct_baseline,total_absolute_error_baseline,error_contribution_pct_baseline,rmsle_improvement_abs,rmsle_improvement_pct,mae_improvement_abs,mae_improvement_pct,wape_improvement_pp,absolute_error_reduction,absolute_error_reduction_pct,abs_total_bias_pct_lightgbm,abs_total_bias_pct_baseline,abs_total_bias_improvement_pp
26,GROCERY I,864.0000,"3,980,412.4840","3,897,033.7463",0.2350,551.5564,0.1197,-96.5032,-0.0209,"476,544.7603",0.2453,864.0000,"3,980,412.4840","3,902,811.7500",0.2216,775.4987,0.1683,-89.8154,-0.0195,"670,030.8994",0.2417,-0.0134,-0.0605,223.9423,0.2888,4.8610,"193,486.1391",0.2888,0.0209,0.0195,-0.1452
30,PRODUCE,864.0000,"1,977,480.3762","2,021,245.3423",0.2014,257.2741,0.1124,50.6539,0.0221,"222,284.8539",0.1144,864.0000,"1,977,480.3762","2,031,502.0000",0.2825,475.9721,0.2080,62.5248,0.0273,"411,239.9177",0.1484,0.0811,0.2870,218.6980,0.4595,9.5553,"188,955.0638",0.4595,0.0221,0.0273,0.5187
23,BEVERAGES,864.0000,"3,005,407.0000","2,992,293.8654",0.2825,552.5162,0.1588,-15.1772,-0.0044,"477,373.9793",0.2457,864.0000,"3,005,407.0000","3,108,517.0000",0.2769,738.2934,0.2122,119.3404,0.0343,"637,885.5100",0.2301,-0.0055,-0.0200,185.7772,0.2516,5.3408,"160,511.5307",0.2516,0.0044,0.0343,2.9945
20,CLEANING,864.0000,"1,065,243.0000","1,083,779.3844",0.3003,248.8984,0.2019,21.4541,0.0174,"215,048.2479",0.1107,864.0000,"1,065,243.0000","1,140,946.7500",0.3670,346.8018,0.2813,87.6202,0.0711,"299,636.7854",0.1081,0.0667,0.1817,97.9034,0.2823,7.9408,"84,588.5375",0.2823,0.0174,0.0711,5.3666
31,DAIRY,864.0000,"721,034.0000","761,418.1835",0.1984,95.8416,0.1148,46.7410,0.0560,"82,807.1303",0.0426,864.0000,"721,034.0000","752,379.4375",0.2168,133.8872,0.1604,36.2794,0.0435,"115,678.5025",0.0417,0.0184,0.0846,38.0456,0.2842,4.5589,"32,871.3722",0.2842,0.0560,0.0435,-1.2536
21,MEATS,864.0000,"321,070.5266","316,533.1595",0.2983,53.3136,0.1435,-5.2516,-0.0141,"46,062.9686",0.0237,864.0000,"321,070.5266","324,616.9062",0.3033,87.9102,0.2366,4.1046,0.0110,"75,954.4170",0.0274,0.0049,0.0163,34.5966,0.3935,9.3099,"29,891.4484",0.3935,0.0141,0.0110,-0.3087
24,POULTRY,864.0000,"323,560.2689","336,278.5597",0.2712,53.3400,0.1424,14.7202,0.0393,"46,085.7573",0.0237,864.0000,"323,560.2689","338,349.4375",0.2897,80.0163,0.2137,17.1171,0.0457,"69,134.0672",0.0249,0.0185,0.0639,26.6763,0.3334,7.1233,"23,048.3098",0.3334,0.0393,0.0457,0.6400
28,PERSONAL CARE,864.0000,"277,631.0000","273,320.3304",0.2259,47.6868,0.1484,-4.9892,-0.0155,"41,201.3613",0.0212,864.0000,"277,631.0000","263,021.6875",0.2869,69.4272,0.2161,-16.9089,-0.0526,"59,985.1432",0.0216,0.0610,0.2126,21.7405,0.3131,6.7657,"18,783.7819",0.3131,0.0155,0.0526,3.7095
27,BREAD/BAKERY,864.0000,"466,795.5009","470,157.3400",0.2316,68.8440,0.1274,3.8910,0.0072,"59,481.2112",0.0306,864.0000,"466,795.5009","466,484.4062",0.2324,87.4727,0.1619,-0.3601,-0.0007,"75,576.4430",0.0273,0.0008,0.0033,18.6287,0.2130,3.4480,"16,095.2318",0.2130,0.0072,0.0007,-0.6536
12,"LIQUOR,WINE,BEER",864.0000,"84,091.0000","77,472.1131",0.4694,26.8886,0.2763,-7.6607,-0.0787,"23,231.7257",0.0120,864.0000,"84,091.0000","80,656.5703",0.6937,45.0312,0.4627,-3.9750,-0.0408,"38,906.9286",0.0140,0.2243,0.3233,18.1426,0.4029,18.6408,"15,675.2029",0.4029,0.0787,0.0408,-3.7869


### Section conclusion

The segment-level comparison confirms that LightGBM v1 delivers a strong improvement over the best baseline, but the improvement is not uniform across all segments.

LightGBM improves RMSLE across all validation dates and horizon days, which supports temporal stability. However, horizon days 13 and 14 show worse WAPE than the baseline, meaning that some high-volume days still require operational review.

The model provides a clear improvement in promotion periods. For rows with promotion, LightGBM reduces RMSLE by 36.46%, improves WAPE by 6.35 percentage points, and reduces total absolute error by 30.62%. This is particularly relevant because promotion rows concentrate most of the validation sales volume.

At store level, LightGBM substantially improves several high-volume stores such as 44, 45, 46, 47, 48, and 51. However, store 18 and store 25 remain critical exceptions where LightGBM performs worse than the baseline, especially due to strong underprediction.

At family level, LightGBM improves `SCHOOL AND OFFICE SUPPLIES` compared with the baseline, but this family remains the most difficult category. High-volume families such as `GROCERY I`, `PRODUCE`, `BEVERAGES`, and `CLEANING` show large reductions in absolute error, which is important for replenishment and planning.

Overall, LightGBM v1 provides meaningful operational value over the best baseline, especially in promotional and high-volume segments. The main remaining risks are specific stores, late-horizon high-volume days, and event-driven or seasonal families.


# 16.Business Interpretation

This section translates the LightGBM v1 results into business implications for store replenishment and short-term planning.

The model is not evaluated only as a technical forecasting exercise. It is also assessed by how useful it would be for identifying operational risk, reducing planning error, and improving forecast reliability across stores, product families, promotions, and the 16-day forecast horizon.


### Business interpretation

LightGBM v1 provides a clear improvement over the best baseline model.

The model reduces RMSLE from 0.5216 to 0.4167, which represents a 20.11% improvement over the previous benchmark. It also improves MAE from 97.22 to 68.14 and reduces WAPE from 20.81% to 14.59%.

From a business perspective, this means the model produces more accurate short-term forecasts and reduces the weighted absolute forecasting error by approximately 6.23 percentage points.

The model also improves total forecast bias. The best baseline overpredicted total validation sales by 1.34%, while LightGBM v1 produces a nearly neutral total bias of -0.08%. This is important for replenishment because a biased forecast can systematically create either excess inventory or stockout risk.

LightGBM v1 is especially valuable in promotional periods. Promotion rows concentrate most of the validation sales volume, and the model reduces total absolute error in promotion rows by 30.62% compared with the best baseline. This suggests that LightGBM is better suited than simple historical averages for handling demand variation during promotion-heavy periods.

The model also improves several high-volume stores and product families. In particular, large absolute error reductions appear in high-volume families such as `GROCERY I`, `PRODUCE`, `BEVERAGES`, and `CLEANING`. These improvements are operationally relevant because they affect product groups with a large impact on replenishment decisions.

However, the model is not uniformly better in every segment. Store 18 and store 25 remain critical exceptions, with strong underprediction and worse performance than the baseline. Late-horizon days, especially days 13 and 14, also show higher WAPE than the baseline despite better RMSLE. This indicates that some high-volume days still require additional review.

`SCHOOL AND OFFICE SUPPLIES` remains the most difficult family. LightGBM improves this family compared with the baseline, but the model still strongly underpredicts it. This suggests that the current feature set does not fully capture its seasonal or event-driven demand pattern.

Overall, LightGBM v1 is a strong first machine learning model for the project. It improves global accuracy, reduces operational error in high-volume and promotional segments, and produces a nearly neutral aggregate bias. The remaining weaknesses are concentrated in specific stores, late-horizon dates, and seasonal or event-driven families.


### Section conclusion

LightGBM v1 provides meaningful business value over the best baseline model.

The model improves the main technical metric, reduces weighted absolute error, and keeps aggregate forecast bias almost neutral. This makes it a strong first candidate for short-term replenishment and planning.

The model should still be used with caution in specific high-risk segments, especially store 18, store 25, late-horizon high-volume days, and `SCHOOL AND OFFICE SUPPLIES`.

These findings define the next modeling priorities: improve difficult stores, better capture seasonal or event-driven demand, and validate the model across additional temporal windows before deployment.


# 17.Save Model Artifacts

This section saves the LightGBM v1 model artifacts.

The saved outputs include the trained model, validation predictions, metrics, baseline comparison tables, feature importance, segment-level error analysis, training diagnostics, and experiment configuration.

These artifacts are designed to support future MLflow tracking, model comparison, reporting, and deployment work.


In [13]:
def make_json_serializable(value):
    """
    Convert common Python, NumPy, pandas, and pathlib objects into JSON-serializable values.
    """
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    if isinstance(value, (np.ndarray,)):
        return value.tolist()

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if isinstance(value, dict):
        return {str(key): make_json_serializable(item) for key, item in value.items()}

    if isinstance(value, (list, tuple)):
        return [make_json_serializable(item) for item in value]

    return value


def save_json(data, path):
    """Save a dictionary as a formatted JSON file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            make_json_serializable(data),
            file,
            indent=4,
            ensure_ascii=False,
        )


def save_dataframe_csv(df, path):
    """Save a DataFrame as CSV."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_dataframe_parquet(df, path):
    """Save a DataFrame as parquet."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


LIGHTGBM_TRAINING_DIAGNOSTICS_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_training_diagnostics.csv"
)

LIGHTGBM_TRAINING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_training_summary.csv"
)

LIGHTGBM_OVERFITTING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_overfitting_summary.csv"
)

LIGHTGBM_LEARNING_CURVE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_learning_curve.csv"
)

LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_feature_group_importance.csv"
)

LIGHTGBM_SEGMENT_COMPARISON_BY_DATE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_segment_comparison_by_date.csv"
)

LIGHTGBM_SEGMENT_COMPARISON_BY_HORIZON_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_segment_comparison_by_horizon_day.csv"
)

LIGHTGBM_SEGMENT_COMPARISON_BY_STORE_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_segment_comparison_by_store.csv"
)

LIGHTGBM_SEGMENT_COMPARISON_BY_FAMILY_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_segment_comparison_by_family.csv"
)

LIGHTGBM_SEGMENT_COMPARISON_BY_PROMOTION_PATH = (
    REPORTS_TABLES_DIR / f"{MODEL_VERSION}_segment_comparison_by_promotion.csv"
)

LIGHTGBM_EXPERIMENT_SUMMARY_PATH = (
    LIGHTGBM_MODELS_DIR / f"{MODEL_VERSION}_experiment_summary.json"
)

lightgbm_v1_model.save_model(
    str(LIGHTGBM_MODEL_PATH),
    num_iteration=best_iteration,
)

save_dataframe_parquet(
    lightgbm_v1_validation_predictions,
    LIGHTGBM_VALIDATION_PREDICTIONS_PATH,
)

save_dataframe_csv(lightgbm_v1_metrics, LIGHTGBM_METRICS_PATH)
save_dataframe_csv(model_comparison, LIGHTGBM_BASELINE_COMPARISON_PATH)
save_dataframe_csv(
    lightgbm_vs_best_baseline,
    LIGHTGBM_MODELS_DIR / f"{MODEL_VERSION}_vs_best_baseline.csv",
)

save_dataframe_csv(lightgbm_v1_training_summary, LIGHTGBM_TRAINING_SUMMARY_PATH)
save_dataframe_csv(training_diagnostics, LIGHTGBM_TRAINING_DIAGNOSTICS_PATH)
save_dataframe_csv(overfitting_summary, LIGHTGBM_OVERFITTING_SUMMARY_PATH)
save_dataframe_csv(learning_curve, LIGHTGBM_LEARNING_CURVE_PATH)

save_dataframe_csv(feature_importance, LIGHTGBM_FEATURE_IMPORTANCE_PATH)
save_dataframe_csv(feature_group_importance, LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH)

save_dataframe_csv(metrics_by_date, LIGHTGBM_METRICS_BY_DATE_PATH)
save_dataframe_csv(metrics_by_store, LIGHTGBM_METRICS_BY_STORE_PATH)
save_dataframe_csv(metrics_by_family, LIGHTGBM_METRICS_BY_FAMILY_PATH)
save_dataframe_csv(metrics_by_promotion, LIGHTGBM_METRICS_BY_PROMOTION_PATH)
save_dataframe_csv(metrics_by_horizon_day, LIGHTGBM_METRICS_BY_HORIZON_PATH)

save_dataframe_csv(segment_comparison_by_date, LIGHTGBM_SEGMENT_COMPARISON_BY_DATE_PATH)
save_dataframe_csv(
    segment_comparison_by_horizon_day, LIGHTGBM_SEGMENT_COMPARISON_BY_HORIZON_PATH
)
save_dataframe_csv(
    segment_comparison_by_store, LIGHTGBM_SEGMENT_COMPARISON_BY_STORE_PATH
)
save_dataframe_csv(
    segment_comparison_by_family, LIGHTGBM_SEGMENT_COMPARISON_BY_FAMILY_PATH
)
save_dataframe_csv(
    segment_comparison_by_promotion, LIGHTGBM_SEGMENT_COMPARISON_BY_PROMOTION_PATH
)

feature_list_artifact = {
    "model_version": MODEL_VERSION,
    "target_column": TARGET_COLUMN,
    "target_transformation": "log1p",
    "n_features": len(model_features),
    "model_features": model_features,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "excluded_columns": BASE_EXCLUDED_COLUMNS,
    "potential_leakage_columns_checked": POTENTIAL_LEAKAGE_COLUMNS,
}

save_json(feature_list_artifact, LIGHTGBM_FEATURE_LIST_PATH)

lightgbm_config_artifact = {
    "model_version": MODEL_VERSION,
    "created_at": datetime.now(),
    "model_type": "LightGBM regression",
    "target_column": TARGET_COLUMN,
    "target_transformation": "log1p(sales)",
    "prediction_inverse_transformation": "expm1(prediction)",
    "negative_prediction_policy": "clip_to_zero",
    "training_period": {
        "start_date": X_train.index.min() if X_train.index is not None else None,
        "rows": X_train.shape[0],
    },
    "validation_period": {
        "start_date": lightgbm_v1_validation_predictions["date"].min(),
        "end_date": lightgbm_v1_validation_predictions["date"].max(),
        "rows": X_valid.shape[0],
    },
    "features": {
        "n_features": len(model_features),
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
    },
    "lightgbm_params": lgb_params,
    "training_controls": {
        "num_boost_round_limit": FULL_TRAINING_BOOST_ROUNDS,
        "best_iteration": best_iteration,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "training_progress_period": TRAINING_PROGRESS_PERIOD,
        "num_threads": NUM_THREADS,
        "use_gpu": USE_GPU,
        "seed": SEED,
    },
    "main_validation_metrics": lightgbm_v1_metrics.iloc[0].to_dict(),
    "best_baseline_comparison": lightgbm_vs_best_baseline.iloc[0].to_dict(),
    "artifact_paths": {
        "model": LIGHTGBM_MODEL_PATH,
        "feature_list": LIGHTGBM_FEATURE_LIST_PATH,
        "config": LIGHTGBM_CONFIG_PATH,
        "validation_predictions": LIGHTGBM_VALIDATION_PREDICTIONS_PATH,
        "metrics": LIGHTGBM_METRICS_PATH,
        "baseline_comparison": LIGHTGBM_BASELINE_COMPARISON_PATH,
        "feature_importance": LIGHTGBM_FEATURE_IMPORTANCE_PATH,
    },
}

save_json(lightgbm_config_artifact, LIGHTGBM_CONFIG_PATH)

experiment_summary = {
    "model_version": MODEL_VERSION,
    "decision": "accepted_as_first_main_machine_learning_model",
    "benchmark_model": lightgbm_vs_best_baseline["benchmark_model"].iloc[0],
    "main_results": {
        "rmsle": lightgbm_v1_metrics["rmsle"].iloc[0],
        "mae": lightgbm_v1_metrics["mae"].iloc[0],
        "wape": lightgbm_v1_metrics["wape"].iloc[0],
        "total_bias_pct": lightgbm_v1_metrics["total_bias_pct"].iloc[0],
        "rmsle_improvement_pct_vs_best_baseline": lightgbm_vs_best_baseline[
            "rmsle_improvement_pct"
        ].iloc[0],
        "wape_improvement_pct_vs_best_baseline": lightgbm_vs_best_baseline[
            "wape_improvement_pct"
        ].iloc[0],
    },
    "main_strengths": [
        "Improves global RMSLE over the best baseline.",
        "Reduces WAPE and MAE substantially.",
        "Keeps aggregate forecast bias close to neutral.",
        "Improves promotional and high-volume segments.",
    ],
    "main_risks": [
        "Store 18 and store 25 remain critical exceptions.",
        "Late-horizon days 13 and 14 show worse WAPE than the baseline.",
        "SCHOOL AND OFFICE SUPPLIES remains difficult and underpredicted.",
    ],
    "mlflow_ready_metadata": {
        "params": lgb_params,
        "metrics": lightgbm_v1_metrics.iloc[0].to_dict(),
        "tags": {
            "model_version": MODEL_VERSION,
            "model_family": "lightgbm",
            "dataset_source": "notebook_03_feature_engineering_baselines",
            "validation_strategy": "temporal_holdout_16_days",
            "target_transformation": "log1p",
        },
        "artifacts": [
            str(LIGHTGBM_MODEL_PATH),
            str(LIGHTGBM_FEATURE_LIST_PATH),
            str(LIGHTGBM_CONFIG_PATH),
            str(LIGHTGBM_VALIDATION_PREDICTIONS_PATH),
            str(LIGHTGBM_METRICS_PATH),
            str(LIGHTGBM_BASELINE_COMPARISON_PATH),
            str(LIGHTGBM_FEATURE_IMPORTANCE_PATH),
        ],
    },
}

save_json(experiment_summary, LIGHTGBM_EXPERIMENT_SUMMARY_PATH)

saved_artifacts = {
    "model": LIGHTGBM_MODEL_PATH,
    "config": LIGHTGBM_CONFIG_PATH,
    "experiment_summary": LIGHTGBM_EXPERIMENT_SUMMARY_PATH,
    "feature_list": LIGHTGBM_FEATURE_LIST_PATH,
    "validation_predictions": LIGHTGBM_VALIDATION_PREDICTIONS_PATH,
    "metrics": LIGHTGBM_METRICS_PATH,
    "baseline_comparison": LIGHTGBM_BASELINE_COMPARISON_PATH,
    "training_summary": LIGHTGBM_TRAINING_SUMMARY_PATH,
    "training_diagnostics": LIGHTGBM_TRAINING_DIAGNOSTICS_PATH,
    "overfitting_summary": LIGHTGBM_OVERFITTING_SUMMARY_PATH,
    "learning_curve": LIGHTGBM_LEARNING_CURVE_PATH,
    "feature_importance": LIGHTGBM_FEATURE_IMPORTANCE_PATH,
    "feature_group_importance": LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH,
    "metrics_by_date": LIGHTGBM_METRICS_BY_DATE_PATH,
    "metrics_by_store": LIGHTGBM_METRICS_BY_STORE_PATH,
    "metrics_by_family": LIGHTGBM_METRICS_BY_FAMILY_PATH,
    "metrics_by_promotion": LIGHTGBM_METRICS_BY_PROMOTION_PATH,
    "metrics_by_horizon_day": LIGHTGBM_METRICS_BY_HORIZON_PATH,
    "segment_comparison_by_date": LIGHTGBM_SEGMENT_COMPARISON_BY_DATE_PATH,
    "segment_comparison_by_horizon_day": LIGHTGBM_SEGMENT_COMPARISON_BY_HORIZON_PATH,
    "segment_comparison_by_store": LIGHTGBM_SEGMENT_COMPARISON_BY_STORE_PATH,
    "segment_comparison_by_family": LIGHTGBM_SEGMENT_COMPARISON_BY_FAMILY_PATH,
    "segment_comparison_by_promotion": LIGHTGBM_SEGMENT_COMPARISON_BY_PROMOTION_PATH,
}

saved_artifacts_inventory = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "path": artifact_path,
            "relative_path": artifact_path.relative_to(PROJECT_ROOT),
            "exists": artifact_path.exists(),
            "file_size_mb": format_file_size_mb(artifact_path),
        }
        for artifact_name, artifact_path in saved_artifacts.items()
    ]
)

display(saved_artifacts_inventory)

missing_saved_artifacts = saved_artifacts_inventory.loc[
    ~saved_artifacts_inventory["exists"]
]

if not missing_saved_artifacts.empty:
    raise FileNotFoundError(
        "Some LightGBM artifacts were not saved correctly:\n"
        + "\n".join(
            [
                f"- {row.artifact}: {row.path}"
                for row in missing_saved_artifacts.itertuples(index=False)
            ]
        )
    )

print(f"Saved {len(saved_artifacts_inventory)} LightGBM artifacts.")
print(f"Main model artifact: {LIGHTGBM_MODEL_PATH.relative_to(PROJECT_ROOT)}")


,artifact,path,relative_path,exists,file_size_mb
0,model,....,models\lightgbm\lightgbm_v1_model.txt,True,3.4300
1,config,....,models\lightgbm\lightgbm_v1_config.json,True,0.0100
2,experiment_summary,....,models\lightgbm\lightgbm_v1_experiment_summary...,True,0.0000
3,feature_list,....,models\lightgbm\lightgbm_v1_feature_list.json,True,0.0000
4,validation_predictions,....,data\predictions\lightgbm_v1_validation_predic...,True,1.5800
5,metrics,....,reports\tables\lightgbm_v1_metrics.csv,True,0.0000
6,baseline_comparison,....,reports\tables\lightgbm_v1_baseline_comparison...,True,0.0000
7,training_summary,....,reports\tables\lightgbm_v1_training_summary.csv,True,0.0000
8,training_diagnostics,....,reports\tables\lightgbm_v1_training_diagnostic...,True,0.0000
9,overfitting_summary,....,reports\tables\lightgbm_v1_overfitting_summary...,True,0.0000


Saved 23 LightGBM artifacts.
Main model artifact: models\lightgbm\lightgbm_v1_model.txt


### Section conclusion

The LightGBM v1 model artifacts have been saved successfully.

The saved outputs include the trained model, validation predictions, evaluation metrics, baseline comparisons, feature importance, segment-level error analysis, training diagnostics, and experiment metadata.

These artifacts make the experiment reproducible and ready for future MLflow tracking, model comparison, reporting, and deployment preparation.


# 18.Final Notebook Conclusion

This section summarizes the main technical and business conclusions from the LightGBM v1 experiment.


### Final conclusion

LightGBM v1 is accepted as the first main machine learning model candidate for the Corporación Favorita Store Sales Forecasting project.

The model clearly improves the best baseline, `baseline_rolling_mean_28_origin`, on the main validation metrics:

* RMSLE improves from 0.5216 to 0.4167;
* MAE improves from 97.22 to 68.14;
* WAPE improves from 20.81% to 14.59%;
* total forecast bias improves from +1.34% to -0.08%.

This means LightGBM v1 improves both technical forecasting accuracy and business-oriented error metrics. The near-neutral aggregate bias is especially important because it reduces the risk of systematic overstocking or understocking at total sales level.

The model is also technically coherent. Feature importance shows that LightGBM mainly relies on safe historical sales features, especially lag and rolling demand variables shifted to avoid future information leakage. Store-product structure, calendar variables, promotions, oil, and holiday/event indicators also contribute to the model.

The overfitting review shows a controlled train-validation gap. The best validation result is reached at iteration 1,174, with a validation RMSE of 0.4167 on the log-transformed target and a moderate train-validation gap of approximately 5.16%.

From a business perspective, LightGBM v1 provides the most value in promotional and high-volume segments. It reduces total absolute error in promotion rows by more than 30% compared with the best baseline, which is relevant because promotions concentrate most of the validation sales volume.

The main remaining risks are concentrated in specific areas:

* store 18 and store 25 show strong underprediction and worse performance than the baseline;
* horizon days 13 and 14 show worse WAPE than the baseline despite improved RMSLE;
* `SCHOOL AND OFFICE SUPPLIES` remains difficult and strongly underpredicted;
* high-volume store-family combinations such as `GROCERY I`, `BEVERAGES`, and `PRODUCE` still explain a large share of total absolute error.

Overall, LightGBM v1 is a strong, reproducible, and defendable first machine learning model. It is suitable as the main model candidate for the next phase of the project, while future iterations should focus on difficult stores, seasonal/event-driven families, additional validation windows, and controlled hyperparameter tuning.


### Notebook closure

This notebook completed the first full LightGBM modeling cycle:

* input artifacts from notebook 03 were validated;
* a leakage-safe feature policy was defined;
* categorical features were handled efficiently;
* a smoke test was executed before full training;
* LightGBM v1 was trained with reproducible parameters;
* validation predictions were generated;
* metrics were calculated on the original sales scale;
* the model was compared against all baselines;
* overfitting and feature importance were reviewed;
* error analysis was performed by date, horizon, store, family, promotion status, and store-family combination;
* LightGBM was compared against the best baseline at segment level;
* model artifacts, metrics, predictions, diagnostics, and metadata were saved.

The notebook is ready to support future MLflow tracking, Prophet comparison, model registry decisions, and deployment preparation.


# 19.Kaggle Submission with LightGBM v1

This optional section generates the Kaggle submission file for LightGBM v1. 

After validating the model, a final LightGBM model is trained using the full available historical dataset. 
The number of boosting rounds is fixed using the best validation iteration found earlier.

This final model is then used to predict the Kaggle test period and create a submission file with the
required ID and Sales columns.


In [14]:
FULL_MODEL_VERSION = f"{MODEL_VERSION}_full"

LIGHTGBM_FULL_MODEL_PATH = LIGHTGBM_MODELS_DIR / f"{FULL_MODEL_VERSION}_model.txt"
LIGHTGBM_FULL_CONFIG_PATH = LIGHTGBM_MODELS_DIR / f"{FULL_MODEL_VERSION}_config.json"

RAW_DIR = DATA_DIR / "raw"
SAMPLE_SUBMISSION_PATH = RAW_DIR / "sample_submission.csv"

if "id" not in test_features.columns:
    raise ValueError("The test feature dataset must contain the 'id' column.")

if not BASELINE_FULL_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Full feature dataset not found: {BASELINE_FULL_FEATURES_PATH}"
    )

if not SAMPLE_SUBMISSION_PATH.exists():
    raise FileNotFoundError(
        f"Sample submission file not found: {SAMPLE_SUBMISSION_PATH}"
    )

objects_to_release = [
    "X_train",
    "X_valid",
    "y_train",
    "y_valid",
    "valid_actuals",
    "valid_predictions",
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

print("Released validation training matrices before loading full features.")

full_features = read_parquet_with_log(
    BASELINE_FULL_FEATURES_PATH,
    "full_features",
)

full_features["date"] = pd.to_datetime(full_features["date"])

required_full_columns = {"date", "store_nbr", "family", TARGET_COLUMN}
missing_full_columns = required_full_columns - set(full_features.columns)

if missing_full_columns:
    raise ValueError(
        f"Missing columns in full_features: {sorted(missing_full_columns)}"
    )

if TARGET_COLUMN in test_features.columns:
    raise ValueError("The test feature dataset should not contain the target column.")

missing_full_model_features = sorted(set(model_features) - set(full_features.columns))

if missing_full_model_features:
    raise ValueError(
        f"Model features missing in full_features: {missing_full_model_features}"
    )

categorical_feature_summary_full = align_categorical_features(
    dataframes=[full_features, test_features],
    categorical_columns=categorical_features,
)

X_full = full_features.loc[:, model_features]
y_full = np.log1p(full_features[TARGET_COLUMN]).astype("float32")
X_test = test_features.loc[:, model_features]

full_matrix_overview = pd.DataFrame(
    [
        {
            "matrix": "X_full",
            "rows": X_full.shape[0],
            "columns": X_full.shape[1],
            "memory_mb": dataframe_memory_mb(X_full),
            "start_date": full_features["date"].min(),
            "end_date": full_features["date"].max(),
        },
        {
            "matrix": "X_test",
            "rows": X_test.shape[0],
            "columns": X_test.shape[1],
            "memory_mb": dataframe_memory_mb(X_test),
            "start_date": test_features["date"].min(),
            "end_date": test_features["date"].max(),
        },
        {
            "matrix": "y_full",
            "rows": len(y_full),
            "columns": 1,
            "memory_mb": round(y_full.memory_usage(deep=True) / (1024**2), 2),
            "start_date": full_features["date"].min(),
            "end_date": full_features["date"].max(),
        },
    ]
)

display(categorical_feature_summary_full)
display(full_matrix_overview)

if (full_features[TARGET_COLUMN] < 0).any():
    raise ValueError("Negative sales values found in full_features.")

if not np.isfinite(y_full).all():
    raise ValueError("Non-finite values found in y_full.")

full_train_dataset = lgb.Dataset(
    X_full,
    label=y_full,
    categorical_feature=categorical_features,
    free_raw_data=False,
)

print(f"Starting full-model training for {best_iteration:,} boosting rounds...")

full_training_start_time = time.perf_counter()
submission_training_progress = LightGBMTqdmCallback(
    total_rounds=best_iteration,
    description="LightGBM full training",
)

try:
    lightgbm_v1_full_model = lgb.train(
        params=lgb_params,
        train_set=full_train_dataset,
        num_boost_round=best_iteration,
        valid_sets=[full_train_dataset],
        valid_names=["full_train"],
        callbacks=[
            submission_training_progress,
            lgb.log_evaluation(period=TRAINING_PROGRESS_PERIOD),
        ],
    )
finally:
    submission_training_progress.close()

full_training_time_seconds = time.perf_counter() - full_training_start_time

lightgbm_v1_full_model.save_model(
    str(LIGHTGBM_FULL_MODEL_PATH),
    num_iteration=best_iteration,
)

print(f"Generating test predictions for {len(X_test):,} rows...")

test_log_predictions = lightgbm_v1_full_model.predict(
    X_test,
    num_iteration=best_iteration,
)

test_predictions = np.expm1(test_log_predictions)
test_predictions = clip_forecast_predictions(test_predictions)

lightgbm_v1_test_predictions = test_features[
    [
        column
        for column in [
            "id",
            "date",
            "store_nbr",
            "family",
            "onpromotion",
        ]
        if column in test_features.columns
    ]
].copy()

lightgbm_v1_test_predictions["model"] = FULL_MODEL_VERSION
lightgbm_v1_test_predictions["sales"] = test_predictions

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

submission = sample_submission[["id"]].merge(
    lightgbm_v1_test_predictions[["id", "sales"]],
    on="id",
    how="left",
)

if submission["sales"].isna().any():
    missing_submission_rows = submission["sales"].isna().sum()
    raise ValueError(
        f"Submission contains {missing_submission_rows:,} missing predictions."
    )

if (submission["sales"] < 0).any():
    raise ValueError("Submission contains negative sales predictions.")

save_dataframe_parquet(
    lightgbm_v1_test_predictions,
    LIGHTGBM_TEST_PREDICTIONS_PATH,
)

save_dataframe_csv(
    submission,
    LIGHTGBM_SUBMISSION_PATH,
)

full_model_config = {
    "model_version": FULL_MODEL_VERSION,
    "base_model_version": MODEL_VERSION,
    "created_at": datetime.now(),
    "purpose": "kaggle_submission",
    "training_dataset": "baseline_full_features.parquet",
    "test_dataset": "baseline_test_features.parquet",
    "target_column": TARGET_COLUMN,
    "target_transformation": "log1p(sales)",
    "prediction_inverse_transformation": "expm1(prediction)",
    "negative_prediction_policy": "clip_to_zero",
    "num_boost_round": best_iteration,
    "lightgbm_params": lgb_params,
    "features": {
        "n_features": len(model_features),
        "model_features": model_features,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
    },
    "training_summary": {
        "training_rows": X_full.shape[0],
        "test_rows": X_test.shape[0],
        "training_start_date": full_features["date"].min(),
        "training_end_date": full_features["date"].max(),
        "test_start_date": test_features["date"].min(),
        "test_end_date": test_features["date"].max(),
        "training_time_seconds": full_training_time_seconds,
        "training_time_minutes": full_training_time_seconds / 60,
    },
    "artifact_paths": {
        "full_model": LIGHTGBM_FULL_MODEL_PATH,
        "test_predictions": LIGHTGBM_TEST_PREDICTIONS_PATH,
        "submission": LIGHTGBM_SUBMISSION_PATH,
        "config": LIGHTGBM_FULL_CONFIG_PATH,
    },
}

save_json(full_model_config, LIGHTGBM_FULL_CONFIG_PATH)

submission_summary = pd.DataFrame(
    [
        {
            "artifact": "lightgbm_v1_full_model",
            "path": LIGHTGBM_FULL_MODEL_PATH.relative_to(PROJECT_ROOT),
            "exists": LIGHTGBM_FULL_MODEL_PATH.exists(),
            "file_size_mb": format_file_size_mb(LIGHTGBM_FULL_MODEL_PATH),
        },
        {
            "artifact": "lightgbm_v1_test_predictions",
            "path": LIGHTGBM_TEST_PREDICTIONS_PATH.relative_to(PROJECT_ROOT),
            "exists": LIGHTGBM_TEST_PREDICTIONS_PATH.exists(),
            "file_size_mb": format_file_size_mb(LIGHTGBM_TEST_PREDICTIONS_PATH),
        },
        {
            "artifact": "lightgbm_v1_submission",
            "path": LIGHTGBM_SUBMISSION_PATH.relative_to(PROJECT_ROOT),
            "exists": LIGHTGBM_SUBMISSION_PATH.exists(),
            "file_size_mb": format_file_size_mb(LIGHTGBM_SUBMISSION_PATH),
        },
        {
            "artifact": "lightgbm_v1_full_config",
            "path": LIGHTGBM_FULL_CONFIG_PATH.relative_to(PROJECT_ROOT),
            "exists": LIGHTGBM_FULL_CONFIG_PATH.exists(),
            "file_size_mb": format_file_size_mb(LIGHTGBM_FULL_CONFIG_PATH),
        },
    ]
)

submission_validation = pd.DataFrame(
    [
        {
            "rows": len(submission),
            "expected_rows": len(sample_submission),
            "unique_ids": submission["id"].nunique(),
            "missing_predictions": submission["sales"].isna().sum(),
            "negative_predictions": (submission["sales"] < 0).sum(),
            "min_prediction": submission["sales"].min(),
            "mean_prediction": submission["sales"].mean(),
            "median_prediction": submission["sales"].median(),
            "max_prediction": submission["sales"].max(),
            "total_predicted_sales": submission["sales"].sum(),
        }
    ]
)

display(submission_summary)
display(submission_validation)
display(submission.head())

del full_train_dataset, test_log_predictions
gc.collect()

print(f"Submission saved to: {LIGHTGBM_SUBMISSION_PATH.relative_to(PROJECT_ROOT)}")


Released validation training matrices before loading full features.
Loading full_features...
Loaded full_features: 3,000,888 rows, 44 columns, 751.29 MB, 3.36 seconds


,feature,n_categories,sample_categories
0,store_nbr,54,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
1,family,33,"[AUTOMOTIVE, BABY CARE, BEAUTY, BEVERAGES, BOO..."
2,city,22,"[Ambato, Babahoyo, Cayambe, Cuenca, Daule, El ..."
3,state,16,"[Azuay, Bolivar, Chimborazo, Cotopaxi, El Oro,..."
4,store_type,5,"[A, B, C, D, E]"
5,cluster,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
6,day_of_week,7,"[0, 1, 2, 3, 4, 5, 6]"
7,month,12,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"


,matrix,rows,columns,memory_mb,start_date,end_date
0,X_full,3000888,41,477.9400,2013-01-01,2017-08-15
1,X_test,28512,41,4.5500,2017-08-16,2017-08-31
2,y_full,3000888,1,11.4500,2013-01-01,2017-08-15


Starting full-model training for 1,174 boosting rounds...


LightGBM full training:   4%|▍         | 51/1174 [00:08<03:03,  6.14round/s, full_train_rmse=0.64093]

...

Generating test predictions for 28,512 rows...


,artifact,path,exists,file_size_mb
0,lightgbm_v1_full_model,models\lightgbm\lightgbm_v1_full_model.txt,True,3.4300
1,lightgbm_v1_test_predictions,data\predictions\lightgbm_v1_test_predictions....,True,0.4500
2,lightgbm_v1_submission,data\predictions\lightgbm_v1_submission.csv,True,0.7500
3,lightgbm_v1_full_config,models\lightgbm\lightgbm_v1_full_config.json,True,0.0000


,rows,expected_rows,unique_ids,missing_predictions,negative_predictions,min_prediction,mean_prediction,median_prediction,max_prediction,total_predicted_sales
0,28512,28512,28512,0,0,0.0000,432.7323,25.8004,"13,715.9870","12,338,063.0894"


,id,sales
0,3000888,4.2664
1,3000889,0.0793
2,3000890,5.1655
3,3000891,"2,419.2533"
4,3000892,0.2286


Submission saved to: data\predictions\lightgbm_v1_submission.csv


### Section conclusion

The LightGBM v1 Kaggle submission has been generated successfully.

A final LightGBM model was retrained using the full available historical dataset and the best validation iteration from the previous experiment. The resulting test predictions were transformed back to the original sales scale, clipped to avoid negative values, and saved in the required Kaggle format.

The submission contains 28,512 rows, matching the expected Kaggle test size. All IDs are unique, there are no missing predictions, and there are no negative sales forecasts.

The generated submission file is available at:

`data/predictions/lightgbm_v1_submission.csv`

The final full-training model and its configuration were also saved under:

`models/lightgbm/`
